
| Stage | Method | Purpose |
|-------|--------|---------|
| 1 | Data loading | KP txt, OMNI dat, DONKI JSON, SWPC JSON |
| 2 | Feature engineering | Lag, rolling, solar/seasonal features |
| 3 | GA | Feature selection via Random Forest CV |
| 4 | XGBoost + PSO | Short-term Kp forecast (1–3 day) |
| 5 | LSTM × 4 + PSO | Medium-term Kp forecast (3–12 day) |
| 6 | Ensemble | R²-weighted blend + final comparison |



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted ✓')
!pip install xgboost scikit-learn tensorflow matplotlib pandas numpy seaborn joblib -q
print('Dependencies ready ✓')


Mounted at /content/drive
Google Drive mounted ✓
Dependencies ready ✓


In [ ]:

import os, json, glob, warnings, time
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              precision_score, recall_score, f1_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Bidirectional, Dense, Dropout,
    Conv1D, MaxPooling1D, Input, MultiHeadAttention, Add,
    LayerNormalization, GlobalAveragePooling1D, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

print(f"XGBoost  : {xgb.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

XGBoost  : 3.2.0
TensorFlow: 2.19.0
GPU available: False


## 1 — Paths & Configuration

In [ ]:

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')

DATA_FOLDER = 'data'      
TRAIN_FOLDER = 'training-data'
OUT_FOLDER   = 'output-data'   

BASE      = DRIVE_ROOT / DATA_FOLDER
TRAIN_DIR = BASE / TRAIN_FOLDER
OUT_ROOT  = BASE / OUT_FOLDER

OUT_CLEAN  = OUT_ROOT / '01_cleaned_data'
OUT_FEAT   = OUT_ROOT / '02_feature_data'
OUT_GA     = OUT_ROOT / '03_ga_feature_selection'
OUT_XGB    = OUT_ROOT / '04_xgboost'
OUT_LSTM   = OUT_ROOT / '05_lstm'
OUT_EVAL   = OUT_ROOT / '06_evaluation'
OUT_MODELS = OUT_ROOT / '07_models'
OUT_PLOTS  = OUT_ROOT / '08_plots'

for d in [OUT_CLEAN, OUT_FEAT, OUT_GA, OUT_XGB, OUT_LSTM,
          OUT_EVAL, OUT_MODELS, OUT_PLOTS]:
    d.mkdir(parents=True, exist_ok=True)


print(f'TRAIN_DIR : {TRAIN_DIR}')
print(f'Exists    : {TRAIN_DIR.exists()}')
print(f'OUT_ROOT  : {OUT_ROOT}')

if TRAIN_DIR.exists():
    files = sorted(TRAIN_DIR.iterdir())
    print(f'\nFiles in training-data/ ({len(files)} total):')
    for ff in files:
        print(f'  {ff.name}')
else:
    print('\nWARNING: TRAIN_DIR not found.')
    print('  1. Make sure Drive is mounted (run the cell above).')
    print(f'  2. Check MyDrive/{DATA_FOLDER}/{TRAIN_FOLDER}/ exists in your Drive.')
    print('  3. Update DATA_FOLDER / TRAIN_FOLDER above if your folder names differ.')


TRAIN_DIR : /content/drive/MyDrive/data/training-data
Exists    : True
OUT_ROOT  : /content/drive/MyDrive/data/output-data

Files in training-data/ (19 total):
  donki-CME.json
  kp-10y.txt
  kp-1932.txt
  kp-1y.txt
  kp-2000.txt
  kp-30.txt
  kp-5y.txt
  kp_latest_auto.txt
  kp_live_full.txt
  kp_live_temp.txt
  omni format.txt
  omni_m_all_years- 2024.dat.txt
  read-me.txt
  read.me.txt
  swpc boulder_k_index_1m.json
  swpc dscovr_mag_1s.json
  swpc f107_cm_flux.json
  swpc geospace_dst_7_day.json
  swpc rtsw_wind_1m.json


In [ ]:

# Aurora thresholds
AURORA_KP_THRESHOLD = 5.0
STORM_LEVELS = {"Quiet":(0,3), "Active":(3,5), "G1":(5,6), "G2":(6,7), "G3+":(7,10)}

LOOKBACK      = 30     # days of history fed to LSTM
FORECAST_DAYS = 10     # future days to predict
TEST_SPLIT    = 0.15
VAL_SPLIT     = 0.15

# GA feature selection
# Budget: 12 pop × 8 gens × 2 CV × RF(50 trees, depth 6) ≈ 192 fits → ~6 min
GA_POP       = 12
GA_GENS      = 8
GA_MUTATION  = 0.20
GA_CROSSOVER = 0.80
GA_PATIENCE  = 3

# PSO hyperparameter tuning
# Budget: 10 particles × 10 iters × 2 CV × XGB(hist) ≈ 200 fits → ~8 min
PSO_PARTICLES = 10
PSO_ITERS     = 10
PSO_INERTIA   = 0.6
PSO_COGNITIVE = 2.0
PSO_SOCIAL    = 2.0
PSO_PATIENCE  = 4

# Misc
SEED         = 42
N_CV_SPLITS  = 2       # 2-fold CV in GA/PSO fitness; 33% faster per evaluation
TARGET_COL   = "Kp_mean"

# Speed controls 
OPT_SAMPLE_ROWS = 5_000  
LSTM_START_YEAR = 2000   
LSTM_UNITS      = 48    


EPOCHS        = 40       # early stopping fires well before this
BATCH_SIZE    = 128
LEARNING_RATE = 1e-3

COLORS = {
    "XGBoost":      "#f59e0b",
    "Stacked LSTM": "#3b82f6",
    "BiLSTM":       "#10b981",
    "CNN-LSTM":     "#f59e0b",
    "Attn-LSTM":    "#8b5cf6",
    "Actual":       "#1e293b",
    "Ensemble":     "#ef4444",
}

print("Configuration loaded ✓")
print(f"  Target: {TARGET_COL}  |  Lookback: {LOOKBACK}d  |  Forecast: {FORECAST_DAYS}d")

Configuration loaded ✓
  Target: Kp_mean  |  Lookback: 30d  |  Forecast: 10d


## 2 — Data Loading

In [ ]:

def parse_kp_file(filepath):
    """
    Parse GFZ Kp/ap/Ap/SN/F10.7 fixed-width ASCII files.
    Format: YYYY MM DD days days_m Bsr dB Kp1..8 ap1..8 Ap SN F107obs F107adj D
    """
    rows, skipped = [], 0
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 24:
                skipped += 1
                continue
            try:
                year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
                kp  = [float(parts[7+i])  for i in range(8)]
                ap  = [float(parts[15+i]) for i in range(8)]
                Ap      = float(parts[23]) if len(parts) > 23 else np.nan
                SN      = float(parts[24]) if len(parts) > 24 else np.nan
                F107obs = float(parts[25]) if len(parts) > 25 else np.nan
                F107adj = float(parts[26]) if len(parts) > 26 else np.nan
                row = {"date": datetime(year, month, day)}
                for i, v in enumerate(kp, 1):  row[f"Kp{i}"] = v
                for i, v in enumerate(ap, 1):  row[f"ap{i}"] = v
                row.update({"Ap": Ap, "SN": SN, "F107obs": F107obs, "F107adj": F107adj})
                rows.append(row)
            except (ValueError, IndexError):
                skipped += 1
    df = (pd.DataFrame(rows).sort_values("date")
            .drop_duplicates("date").reset_index(drop=True))
    print(f"  KP: {len(df):,} rows parsed, {skipped} skipped  [{filepath.name}]")
    return df


def load_omni(filepath):
    """
    Load OMNI monthly .dat file (omni2 format).
    Positional cols: 0=year 1=doy 2=hr 3=Bx 4=By_GSE 5=Bz_GSE
                     6=By_GSM 7=Bz_GSM 8=|B| 9=speed 10=density 11=temp
    """
    rows = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 13:
                continue
            try:
                year, doy = int(parts[0]), int(parts[1])
                Bx      = float(parts[3]);  By_GSE = float(parts[4])
                Bz_GSE  = float(parts[5]);  By_GSM = float(parts[6])
                Bz_GSM  = float(parts[7]);  B_mag  = float(parts[8])
                speed   = float(parts[9])
                density = float(parts[10]) if len(parts) > 10 else np.nan
                temp    = float(parts[11]) if len(parts) > 11 else np.nan
                date    = datetime.strptime(f"{year} {doy}", "%Y %j")
                rows.append({"date": date, "Bx": Bx, "By_GSE": By_GSE,
                              "Bz_GSE": Bz_GSE, "By_GSM": By_GSM, "Bz_GSM": Bz_GSM,
                              "B_mag": B_mag, "sw_speed": speed,
                              "sw_density": density, "sw_temp": temp})
            except (ValueError, IndexError):
                continue
    if not rows:
        return pd.DataFrame()
    df = (pd.DataFrame(rows).sort_values("date")
            .drop_duplicates("date").reset_index(drop=True))
    fill_vals = [999.9, 9999.9, 99999.9, 9999999.0, 99.99, 999.99]
    for col in ["Bx","By_GSE","Bz_GSE","By_GSM","Bz_GSM","B_mag",
                "sw_speed","sw_density","sw_temp"]:
        for fv in fill_vals:
            df[col] = df[col].replace(fv, np.nan)
        df[col] = df[col].where(df[col].abs() < 9000, np.nan)
    print(f"  OMNI: {len(df):,} rows  [{filepath.name}]")
    return df


def load_donki_cme(filepath):
    """Load DONKI CME JSON — extract daily CME flag and max speed."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    rows = []
    for event in data:
        try:
            start = pd.to_datetime(event.get("startTime", ""))
            if pd.isna(start):
                continue
            speed = np.nan
            for a in (event.get("cmeAnalyses") or []):
                if a and a.get("speed"):
                    speed = float(a["speed"]); break
            rows.append({"date": start.date(), "cme_speed": speed})
        except Exception:
            continue
    if not rows:
        return pd.DataFrame(columns=["date", "cme_speed"])
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    df = df.groupby("date").agg(cme_speed=("cme_speed","max")).reset_index()
    df["cme_flag"] = 1
    print(f"  DONKI CME: {len(df):,} CME days  [{filepath.name}]")
    return df


def load_swpc_json(filepath):
    """Generic SWPC JSON loader — infers parameter name from filename."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    name = filepath.stem.lower()
    rows = []
    if isinstance(data, list):
        for item in data:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                try:
                    rows.append({"timestamp": pd.to_datetime(item[0]),
                                 "value": float(item[1])})
                except Exception:
                    continue
            elif isinstance(item, dict):
                ts_key = next((k for k in item
                               if "time" in k.lower() or "date" in k.lower()), None)
                if not ts_key:
                    continue
                try:
                    ts = pd.to_datetime(item[ts_key])
                except Exception:
                    continue
                for k, v in item.items():
                    if k == ts_key:
                        continue
                    try:
                        rows.append({"timestamp": ts, "value": float(v)}); break
                    except Exception:
                        continue
    if not rows:
        return {}
    df = pd.DataFrame(rows).dropna()
    df["date"] = df["timestamp"].dt.floor("D")
    if   "kp"    in name or "k_index" in name: param = "swpc_kp"
    elif "dst"   in name:                       param = "swpc_dst"
    elif "wind"  in name or "rtsw"    in name:  param = "swpc_sw_speed"
    elif "f107"  in name or "flux"    in name:  param = "swpc_f107"
    elif "dscovr" in name or "mag"   in name:   param = "swpc_bz"
    else:                                        param = "swpc_" + filepath.stem
    daily = df.groupby("date")["value"].mean().reset_index()
    daily.columns = ["date", param]
    daily["date"] = pd.to_datetime(daily["date"])
    print(f"  SWPC JSON: {len(daily):,} daily rows  [{filepath.name}] → {param}")
    return {param: daily}

print("Parsers defined ✓")

Parsers defined ✓


In [ ]:

t0 = time.time()

print("=" * 60)
print("  LOADING DATA")
print("=" * 60)

# KP files
# Matches: kp-1y.txt  kp-5y.txt  kp-10y.txt  kp-30.txt  kp-1932.txt  kp-2000.txt
SKIP_WORDS = ['omni', 'readme', 'read-me', 'read_me', 'format',
               'sunspot', 'radio', 'flux', 'solar']

kp_dfs        = []
kp_candidates = sorted(TRAIN_DIR.glob('kp*.txt')) + sorted(TRAIN_DIR.glob('kp-*.txt'))
kp_candidates = sorted(set(kp_candidates))  # deduplicate

if not kp_candidates:
    # Fallback: try every .txt in the folder that doesn't look like OMNI/readme
    kp_candidates = [f for f in sorted(TRAIN_DIR.glob('*.txt'))
                     if not any(w in f.name.lower() for w in SKIP_WORDS)]
    print(f'  No kp*.txt found -- trying {len(kp_candidates)} other .txt file(s)')

for f in kp_candidates:
    print(f'  Trying: {f.name} ...', end=' ')
    try:
        df = parse_kp_file(f)
        if not df.empty:
            kp_dfs.append(df)
            print(f'  {len(df):,} rows')
        else:
            print('0 rows parsed')
    except Exception as e:
        print(f'WARNING: {e}')

if not kp_dfs:
    print('\nFiles visible in TRAIN_DIR:')
    for ff in sorted(TRAIN_DIR.iterdir()):
        print(f'  {ff.name}')
    raise ValueError(
        'No KP data loaded. Check TRAIN_DIR in the Paths cell '
        'and confirm the folder contains GFZ-format kp*.txt files.')

kp_all = (pd.concat(kp_dfs).sort_values("date")
            .drop_duplicates("date").reset_index(drop=True))
print(f"  KP combined: {len(kp_all):,} rows  "
      f"({kp_all['date'].min().date()} → {kp_all['date'].max().date()})")

# OMNI
omni_all = pd.DataFrame()
# OMNI filename: "omni_m_all_years- 2024.dat.txt"  (note the space before 2024)
omni_files = (list(TRAIN_DIR.glob("omni*.dat.txt")) +
              list(TRAIN_DIR.glob("omni*.dat")) +
              list(TRAIN_DIR.glob("omni*.txt")))
for f in sorted(set(omni_files)):
    if "format" in f.name.lower():
        continue
    try:
        df = load_omni(f)
        if not df.empty:
            omni_all = (pd.concat([omni_all, df])
                          .drop_duplicates("date").sort_values("date"))
    except Exception as e:
        print(f"  WARNING: {f.name}: {e}")

# DONKI CME
cme_df = pd.DataFrame()
# donki-CME.json  (capital CME — case-insensitive match)
for f in sorted(list(TRAIN_DIR.glob("donki*.json")) + list(TRAIN_DIR.glob("Donki*.json")) + list(TRAIN_DIR.glob("DONKI*.json"))):
    try:
        cme_df = load_donki_cme(f)
    except Exception as e:
        print(f"  WARNING: {f.name}: {e}")

# SWPC JSON
swpc_dfs = {}
# SWPC files may have spaces in names e.g. "swpc boulder_k_index_1m.json"
swpc_files = list(TRAIN_DIR.glob("swpc*.json")) + list(TRAIN_DIR.glob("swpc *.json"))
for f in sorted(set(swpc_files)):
    try:
        swpc_dfs.update(load_swpc_json(f))
    except Exception as e:
        print(f"  WARNING: {f.name}: {e}")

# Merge
merged = kp_all.copy()
merged["date"] = pd.to_datetime(merged["date"])

if not omni_all.empty:
    omni_all["date"] = pd.to_datetime(omni_all["date"])
    merged = merged.merge(omni_all, on="date", how="left")

if not cme_df.empty:
    cme_df["date"] = pd.to_datetime(cme_df["date"])
    merged = merged.merge(cme_df, on="date", how="left")
    merged["cme_flag"]  = merged["cme_flag"].fillna(0)
    merged["cme_speed"] = merged["cme_speed"].fillna(0)

for param, df in swpc_dfs.items():
    df["date"] = pd.to_datetime(df["date"])
    merged = merged.merge(df, on="date", how="left")

df_raw = merged.sort_values("date").reset_index(drop=True)
df_raw.to_csv(OUT_CLEAN / "merged_raw.csv", index=False)

print(f"\n  Merged: {len(df_raw):,} rows × {len(df_raw.columns)} cols")
print(f"  Saved:  {OUT_CLEAN / 'merged_raw.csv'}")
print(f"  ⏱  Data loaded in {time.time()-t0:.1f}s")
df_raw.head(3)

  LOADING DATA
  Trying: kp-10y.txt ...   KP: 3,699 rows parsed, 0 skipped  [kp-10y.txt]
  3,699 rows
  Trying: kp-1932.txt ...   KP: 34,380 rows parsed, 0 skipped  [kp-1932.txt]
  34,380 rows
  Trying: kp-1y.txt ...   KP: 411 rows parsed, 0 skipped  [kp-1y.txt]
  411 rows
  Trying: kp-2000.txt ...   KP: 9,543 rows parsed, 0 skipped  [kp-2000.txt]
  9,543 rows
  Trying: kp-30.txt ...   KP: 30 rows parsed, 0 skipped  [kp-30.txt]
  30 rows
  Trying: kp-5y.txt ...   KP: 2,238 rows parsed, 0 skipped  [kp-5y.txt]
  2,238 rows
  Trying: kp_latest_auto.txt ...   KP: 34,416 rows parsed, 0 skipped  [kp_latest_auto.txt]
  34,416 rows
  Trying: kp_live_full.txt ...   KP: 34,416 rows parsed, 0 skipped  [kp_live_full.txt]
  34,416 rows
  Trying: kp_live_temp.txt ...   KP: 34,416 rows parsed, 0 skipped  [kp_live_temp.txt]
  34,416 rows
  KP combined: 34,416 rows  (1932-01-01 → 2026-03-23)
  OMNI: 22,646 rows  [omni_m_all_years- 2024.dat.txt]
  DONKI CME: 31 CME days  [donki-CME.json]
  SWPC JSON: 2 

,date,Kp1,Kp2,Kp3,Kp4,Kp5,Kp6,Kp7,Kp8,ap1,...,sw_speed,sw_density,sw_temp,cme_speed,cme_flag,swpc_kp,swpc_bz,swpc_f107,swpc_dst,swpc_sw_speed
0,1932-01-01,3.333,2.667,2.333,2.667,3.333,2.667,3.333,3.333,18.0,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,1932-01-02,3.667,3.667,3.333,3.667,3.333,4.667,3.000,5.000,22.0,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,1932-01-03,3.333,3.333,3.000,1.000,2.333,1.667,2.667,2.000,18.0,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## 3 — Feature Engineering

In [ ]:

t1 = time.time()

def engineer_features(df):
    df = df.copy()
    kp_cols = [f"Kp{i}" for i in range(1, 9)]
    ap_cols = [f"ap{i}" for i in range(1, 9)]

    for col in kp_cols + ap_cols + ["Ap","SN","F107obs","F107adj"]:
        if col in df.columns:
            df[col] = df[col].replace(-1, np.nan).replace(-1.0, np.nan)

    # Daily Kp stats
    df["Kp_mean"]  = df[kp_cols].mean(axis=1)
    df["Kp_max"]   = df[kp_cols].max(axis=1)
    df["Kp_min"]   = df[kp_cols].min(axis=1)
    df["Kp_std"]   = df[kp_cols].std(axis=1)
    df["Kp_range"] = df["Kp_max"] - df["Kp_min"]

    # Daily ap stats
    df["ap_mean"] = df[ap_cols].mean(axis=1)
    df["ap_max"]  = df[ap_cols].max(axis=1)
    df["ap_sum"]  = df[ap_cols].sum(axis=1)

    # Aurora flags
    df["aurora_flag"]  = (df["Kp_max"] >= AURORA_KP_THRESHOLD).astype(float)
    df["storm_hours"]  = (df[kp_cols] >= AURORA_KP_THRESHOLD).sum(axis=1) * 3
    def storm_class(kp):
        if kp >= 7: return 5
        elif kp >= 6: return 4
        elif kp >= 5: return 3
        elif kp >= 3: return 2
        elif kp >= 2: return 1
        else: return 0
    df["storm_class"] = df["Kp_max"].apply(storm_class)

    # Lag features
    for lag in [1,2,3,7,14,27]:
        df[f"Kp_mean_lag{lag}"] = df["Kp_mean"].shift(lag)
        df[f"Kp_max_lag{lag}"]  = df["Kp_max"].shift(lag)
        df[f"aurora_lag{lag}"]  = df["aurora_flag"].shift(lag)
        if "Ap" in df.columns:
            df[f"Ap_lag{lag}"] = df["Ap"].shift(lag)

    if "Bz_GSM" in df.columns:
        for lag in [1,2,3,7]:
            df[f"Bz_lag{lag}"] = df["Bz_GSM"].shift(lag)
        df["Bz_min_3d"]  = df["Bz_GSM"].rolling(3, min_periods=1).min()
        df["Bz_mean_7d"] = df["Bz_GSM"].rolling(7, min_periods=1).mean()

    if "sw_speed" in df.columns:
        for lag in [1,2,3]:
            df[f"speed_lag{lag}"] = df["sw_speed"].shift(lag)
        df["speed_mean_7d"] = df["sw_speed"].rolling(7, min_periods=1).mean()

    # Rolling statistics
    if "Ap" in df.columns:
        df["Ap_rolling7"]  = df["Ap"].rolling(7,  min_periods=1).mean()
        df["Ap_rolling27"] = df["Ap"].rolling(27, min_periods=1).mean()
    df["Kp_rolling7"]    = df["Kp_mean"].rolling(7,  min_periods=1).mean()
    df["Kp_rolling27"]   = df["Kp_mean"].rolling(27, min_periods=1).mean()
    if "SN" in df.columns:
        df["SN_rolling27"]   = df["SN"].rolling(27, min_periods=1).mean()
    if "F107obs" in df.columns:
        df["F107_rolling27"] = df["F107obs"].rolling(27, min_periods=1).mean()
        df["F107_smooth"]    = df["F107obs"].rolling(27, min_periods=1).mean()

    # Solar / seasonal
    df["year_frac"]         = df["date"].dt.year + df["date"].dt.dayofyear / 365.25
    df["solar_cycle_phase"] = np.sin(2*np.pi*(df["year_frac"] - 1996.5) / 11.3)
    doy = df["date"].dt.dayofyear
    df["season_sin"]        = np.sin(2*np.pi*doy / 365.25)
    df["season_cos"]        = np.cos(2*np.pi*doy / 365.25)
    df["month"]             = df["date"].dt.month
    df["equinox_prox"]      = np.minimum(np.abs(doy-80), np.abs(doy-266))
    df["equinox_weight"]    = np.exp(-df["equinox_prox"] / 20)

    # CME rolling
    if "cme_flag" in df.columns:
        df["cme_3d"] = df["cme_flag"].rolling(3, min_periods=1).sum()
        df["cme_7d"] = df["cme_flag"].rolling(7, min_periods=1).sum()

    df = df.ffill().bfill()
    df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    return df


def get_feature_cols(df):
    exclude = {"date", TARGET_COL, "Kp_max", "Kp_min", "aurora_flag",
               "storm_class", "storm_hours", "year_frac", "F107adj"}
    return [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]


df = engineer_features(df_raw)
df.to_csv(OUT_FEAT / "feature_engineered.csv", index=False)
feature_cols = get_feature_cols(df)

print(f"Feature engineering complete: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Input feature pool: {len(feature_cols)} columns")
print(f"  ⏱  {time.time()-t1:.1f}s")
df[feature_cols[:5]].describe().round(3)

Feature engineering complete: 34,416 rows × 98 cols
Input feature pool: 89 columns
  ⏱  6.0s


,Kp1,Kp2,Kp3,Kp4,Kp5
count,34416.000,34416.000,34416.000,34416.000,34416.000
mean,2.271,2.236,2.098,2.060,2.129
std,1.490,1.494,1.444,1.381,1.407
min,0.000,0.000,0.000,0.000,0.000
25%,1.000,1.000,1.000,1.000,1.000
50%,2.000,2.000,2.000,1.667,2.000
75%,3.333,3.333,3.000,3.000,3.000
max,9.000,9.000,9.000,9.000,9.000


## 4 — Exploratory Data Analysis

In [ ]:

plot_df = df[df["date"] >= "1990-01-01"]
fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)
fig.suptitle("Aurora & Solar Activity Overview", fontsize=16, fontweight="bold")

ax = axes[0]
ax.plot(plot_df["date"], plot_df["Kp_mean"], lw=0.6, color="#3b82f6", label="Kp daily mean")
ax.axhline(5, color="#ef4444", lw=1, ls="--", label="G1 storm threshold (Kp=5)")
ax.fill_between(plot_df["date"], 0, plot_df["Kp_mean"],
                where=plot_df["Kp_mean"] >= 5, alpha=0.25, color="#ef4444", label="Storm events")
ax.set_ylabel("Kp Index"); ax.set_ylim(0, 9.5); ax.legend(fontsize=8)
ax.set_title("Daily Mean Kp (Aurora Proxy)", fontsize=11)

ax = axes[1]
if "Ap" in df.columns:
    ax.plot(plot_df["date"], plot_df["Ap"], lw=0.6, color="#10b981")
ax.set_ylabel("Ap (nT)"); ax.set_title("Daily Equivalent Planetary Amplitude", fontsize=11)

ax = axes[2]
if "SN" in df.columns:
    ax.plot(plot_df["date"], plot_df["SN"], lw=0.4, color="#f59e0b", alpha=0.5, label="Daily SN")
    if "SN_rolling27" in plot_df.columns:
        ax.plot(plot_df["date"], plot_df["SN_rolling27"], lw=1.2, color="#d97706", label="27-day mean")
    ax.legend(fontsize=8)
ax.set_ylabel("Sunspot Number"); ax.set_title("International Sunspot Number", fontsize=11)

ax = axes[3]
if "F107obs" in df.columns:
    ax.plot(plot_df["date"], plot_df["F107obs"], lw=0.6, color="#8b5cf6")
ax.set_ylabel("F10.7 (sfu)"); ax.set_xlabel("Date")
ax.set_title("F10.7 Solar Radio Flux", fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT_PLOTS / "eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()

aurora_days = df["aurora_flag"].sum()
print(f"Total days   : {len(df):,}")
print(f"Aurora days  : {int(aurora_days):,}  ({100*aurora_days/len(df):.1f}%)")
print(f"G2+ days     : {int((df['Kp_max']>=6).sum()):,}")
print(f"G3+ days     : {int((df['Kp_max']>=7).sum()):,}")

Total days   : 34,416
Aurora days  : 5,564  (16.2%)
G2+ days     : 2,142
G3+ days     : 766


## 5 — GA Feature Selection

In [ ]:
class GeneticAlgorithmFS:
    """
    Binary GA over feature mask.
    Fitness = TimeSeriesSplit CV-MAE of a lightweight Random Forest.
    Early stopping on no-improvement for GA_PATIENCE generations.
    """
    def __init__(self):
        self.pop_size      = GA_POP
        self.generations   = GA_GENS
        self.mutation_rate = GA_MUTATION
        self.crossover_rate= GA_CROSSOVER
        self.patience      = GA_PATIENCE
        self.best_mask     = None
        self.best_fitness  = float("inf")
        self.history       = []

    def _random_individual(self, n):
        k    = np.random.randint(3, max(4, n // 2))
        mask = np.zeros(n, dtype=int)
        mask[np.random.choice(n, k, replace=False)] = 1
        return mask

    def _fitness(self, mask, X, y):
        if mask.sum() < 1:
            return float("inf")
        X_sel = X[:, mask.astype(bool)]
        rf    = RandomForestRegressor(
                    n_estimators=50, max_depth=6,
                    max_samples=0.6, n_jobs=-1, random_state=SEED)
        tscv  = TimeSeriesSplit(n_splits=N_CV_SPLITS)
        maes  = [mean_absolute_error(y[va], rf.fit(X_sel[tr], y[tr]).predict(X_sel[va]))
                 for tr, va in tscv.split(X_sel)]
        return float(np.mean(maes)) + 0.001 * mask.sum()

    def _crossover(self, p1, p2):
        if np.random.random() > self.crossover_rate:
            return deepcopy(p1), deepcopy(p2)
        pt = np.random.randint(1, len(p1))
        return (np.concatenate([p1[:pt], p2[pt:]]),
                np.concatenate([p2[:pt], p1[pt:]]))

    def _mutate(self, ind):
        ind = deepcopy(ind)
        for i in range(len(ind)):
            if np.random.random() < self.mutation_rate:
                ind[i] = 1 - ind[i]
        if ind.sum() == 0:
            ind[np.random.randint(len(ind))] = 1
        return ind

    def _tournament(self, pop, scores, k=3):
        idx = np.random.choice(len(pop), k, replace=False)
        return pop[idx[np.argmin([scores[i] for i in idx])]]

    def run(self, X, y, feature_names):
        n = X.shape[1]
        if OPT_SAMPLE_ROWS and len(y) > OPT_SAMPLE_ROWS:
            X, y = X[-OPT_SAMPLE_ROWS:], y[-OPT_SAMPLE_ROWS:]
            print(f"    GA subsampled to {OPT_SAMPLE_ROWS:,} rows")

        pop = [self._random_individual(n) for _ in range(self.pop_size)]
        no_improve = 0
        print(f"  GA  (pop={self.pop_size}, gens={self.generations})")

        for gen in range(self.generations):
            scores = [self._fitness(ind, X, y) for ind in pop]
            best_i = int(np.argmin(scores))
            if scores[best_i] < self.best_fitness:
                self.best_fitness = scores[best_i]
                self.best_mask    = deepcopy(pop[best_i])
                no_improve        = 0
                print(f"    Gen {gen+1:>2}: MAE={self.best_fitness:.4f}  "
                      f"features={self.best_mask.sum()}")
            else:
                no_improve += 1
                print(f"    Gen {gen+1:>2}: no improvement ({no_improve}/{self.patience})")
            self.history.append({"generation": gen,
                                  "best_mae": self.best_fitness,
                                  "mean_mae": float(np.mean(scores))})
            if no_improve >= self.patience:
                print(f"    Early stop at gen {gen+1}")
                break
            new_pop = [deepcopy(pop[best_i])]
            while len(new_pop) < self.pop_size:
                c1, c2 = self._crossover(self._tournament(pop, scores),
                                         self._tournament(pop, scores))
                new_pop += [self._mutate(c1), self._mutate(c2)]
            pop = new_pop[:self.pop_size]

        selected = [feature_names[i] for i, v in enumerate(self.best_mask) if v]
        print(f"\n  GA done — selected {len(selected)}/{n} features:")
        print(f"  {selected}")
        return self.best_mask.astype(bool), selected, self.best_fitness

print("GA class defined ✓")

GA class defined ✓


In [ ]:

t2 = time.time()
print("=" * 60)
print("  STAGE 3 — GA FEATURE SELECTION")
print("=" * 60)

X_ga = df[feature_cols].values.astype(np.float32)
y_ga = df[TARGET_COL].values.astype(np.float32)

ga  = GeneticAlgorithmFS()
ga_mask, selected_features, ga_mae = ga.run(X_ga, y_ga, feature_cols)

# Save
pd.DataFrame(ga.history).to_csv(OUT_GA / "ga_convergence.csv", index=False)
pd.DataFrame({"feature": feature_cols,
              "selected": ga_mask.astype(int)}
             ).to_csv(OUT_GA / "ga_selected_features.csv", index=False)

# Convergence plot
hist_df = pd.DataFrame(ga.history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_df["generation"], hist_df["best_mae"], "o-", color="#3b82f6", label="Best MAE")
ax.plot(hist_df["generation"], hist_df["mean_mae"], "--",  color="#94a3b8",
        label="Mean MAE", alpha=0.7)
ax.set_xlabel("Generation"); ax.set_ylabel("CV-MAE")
ax.set_title("GA Feature Selection — Convergence"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_PLOTS / "ga_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n  GA best CV-MAE : {ga_mae:.4f}")
print(f"  Selected features ({len(selected_features)}): {selected_features}")
print(f"  ⏱  {(time.time()-t2)/60:.1f} min")

  STAGE 3 — GA FEATURE SELECTION
    GA subsampled to 5,000 rows
  GA  (pop=12, gens=8)
    Gen  1: MAE=0.0828  features=31
    Gen  2: no improvement (1/3)
    Gen  3: no improvement (2/3)
    Gen  4: no improvement (3/3)
    Early stop at gen 4

  GA done — selected 31/89 features:
  ['Kp2', 'Kp6', 'ap2', 'ap8', 'Ap', 'SN', 'F107obs', 'Bx', 'By_GSE', 'Bz_GSM', 'B_mag', 'swpc_kp', 'Kp_std', 'ap_mean', 'aurora_lag1', 'Kp_max_lag2', 'Kp_mean_lag3', 'aurora_lag3', 'Ap_lag3', 'Kp_mean_lag7', 'aurora_lag7', 'Kp_mean_lag14', 'Kp_mean_lag27', 'Bz_min_3d', 'Bz_mean_7d', 'speed_lag3', 'speed_mean_7d', 'Ap_rolling7', 'F107_rolling27', 'F107_smooth', 'equinox_prox']

  GA best CV-MAE : 0.0828
  Selected features (31): ['Kp2', 'Kp6', 'ap2', 'ap8', 'Ap', 'SN', 'F107obs', 'Bx', 'By_GSE', 'Bz_GSM', 'B_mag', 'swpc_kp', 'Kp_std', 'ap_mean', 'aurora_lag1', 'Kp_max_lag2', 'Kp_mean_lag3', 'aurora_lag3', 'Ap_lag3', 'Kp_mean_lag7', 'aurora_lag7', 'Kp_mean_lag14', 'Kp_mean_lag27', 'Bz_min_3d', 'Bz_mean_7d',

## 6 — PSO + XGBoost (short-term forecast)

In [ ]:


class PSOHyperparamOptimiser:
    """
    Continuous PSO over XGBoost hyperparameter space.
    Fitness = TimeSeriesSplit CV-MAE on OPT_SAMPLE_ROWS recent rows.
    """
    def __init__(self, search_space):
        self.space    = search_space
        self.n_p      = PSO_PARTICLES
        self.n_i      = PSO_ITERS
        self.w        = PSO_INERTIA
        self.c1       = PSO_COGNITIVE
        self.c2       = PSO_SOCIAL
        self.patience = PSO_PATIENCE
        self.history  = []
        self.best_pos = None
        self.best_fit = float("inf")

    def _decode(self, pos):
        return {k: int(round(float(np.clip(pos[i], lo, hi)))) if is_int
                else float(np.clip(pos[i], lo, hi))
                for i, (k, (lo, hi, is_int)) in enumerate(self.space.items())}

    def _fitness(self, pos, X, y):
        try:
            model = xgb.XGBRegressor(**self._decode(pos), random_state=SEED,
                                      verbosity=0, eval_metric="mae",
                                      tree_method="hist", device="cpu")
            tscv  = TimeSeriesSplit(n_splits=N_CV_SPLITS)
            maes  = []
            for tr, va in tscv.split(X):
                model.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
                maes.append(mean_absolute_error(y[va], model.predict(X[va])))
            return float(np.mean(maes))
        except Exception:
            return float("inf")

    def run(self, X, y):
        if OPT_SAMPLE_ROWS and len(y) > OPT_SAMPLE_ROWS:
            X, y = X[-OPT_SAMPLE_ROWS:], y[-OPT_SAMPLE_ROWS:]
            print(f"    PSO subsampled to {OPT_SAMPLE_ROWS:,} rows")
        n_dim      = len(self.space)
        bounds_lo  = np.array([v[0] for v in self.space.values()], dtype=float)
        bounds_hi  = np.array([v[1] for v in self.space.values()], dtype=float)
        positions  = np.random.uniform(bounds_lo, bounds_hi, (self.n_p, n_dim))
        velocities = np.random.uniform(-(bounds_hi-bounds_lo)*0.1,
                                        (bounds_hi-bounds_lo)*0.1, (self.n_p, n_dim))
        pbest_pos  = positions.copy()
        pbest_fit  = np.full(self.n_p, float("inf"))
        no_improve = 0
        print(f"  PSO  (particles={self.n_p}, iters={self.n_i})")

        for it in range(self.n_i):
            for i in range(self.n_p):
                fit = self._fitness(positions[i], X, y)
                if fit < pbest_fit[i]:
                    pbest_fit[i] = fit; pbest_pos[i] = positions[i].copy()
                if fit < self.best_fit:
                    self.best_fit = fit; self.best_pos = positions[i].copy()
            self.history.append({"iter": it, "best_fit": self.best_fit,
                                  "mean_fit": float(pbest_fit.mean())})
            if it > 0 and self.history[-1]["best_fit"] >= self.history[-2]["best_fit"]:
                no_improve += 1
            else:
                no_improve = 0
            print(f"    Iter {it+1:>3}: best MAE={self.best_fit:.4f}")
            if no_improve >= self.patience:
                print(f"    Early stop at iter {it+1}"); break
            r1 = np.random.rand(self.n_p, n_dim)
            r2 = np.random.rand(self.n_p, n_dim)
            velocities = (self.w  * velocities
                         + self.c1 * r1 * (pbest_pos - positions)
                         + self.c2 * r2 * (self.best_pos - positions))
            positions  = np.clip(positions + velocities, bounds_lo, bounds_hi)

        best_params = self._decode(self.best_pos)
        print(f"\n  PSO best MAE : {self.best_fit:.4f}")
        print(f"  Best params  : {best_params}")
        return best_params, self.best_fit


# XGBoost search space 
XGB_SEARCH_SPACE = {
    "n_estimators":     (150, 400,  True),
    "max_depth":        (3,   6,    True),
    "learning_rate":    (0.03, 0.15, False),
    "subsample":        (0.7,  1.0,  False),
    "colsample_bytree": (0.7,  1.0,  False),
    "min_child_weight": (1,    8,    True),
    "reg_lambda":       (0.5,  2.0,  False),
}

print("PSO class defined ✓")

PSO class defined ✓


In [ ]:
def _compute_metrics(name, y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100)
    t_au = (y_true >= AURORA_KP_THRESHOLD).astype(int)
    p_au = (y_pred >= AURORA_KP_THRESHOLD).astype(int)
    return {"model": name,
            "rmse": round(rmse,4), "mae": round(mae,4),
            "r2": round(r2,4),     "mape": round(mape,2),
            "aurora_precision": round(float(precision_score(t_au, p_au, zero_division=0)),3),
            "aurora_recall":    round(float(recall_score(t_au,    p_au, zero_division=0)),3),
            "aurora_f1":        round(float(f1_score(t_au,        p_au, zero_division=0)),3)}

def _print_metrics(m):
    print(f"    RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  R²={m['r2']:.4f}  "
          f"MAPE={m['mape']:.1f}%  |  Aurora Prec={m['aurora_precision']:.3f}  "
          f"Rec={m['aurora_recall']:.3f}  F1={m['aurora_f1']:.3f}")

def _plot_predictions(dates, y_true, y_pred, title, color, savepath):
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(dates, y_true, lw=1.2, color="#1e293b", label="Actual Kp",    alpha=0.9)
    ax.plot(dates, y_pred, lw=1.0, color=color,     label="Predicted Kp", alpha=0.85, ls="--")
    ax.axhline(AURORA_KP_THRESHOLD, color="#dc2626", lw=0.8, ls=":", alpha=0.6)
    ax.fill_between(dates, AURORA_KP_THRESHOLD, 9.5,
                    where=(y_true >= AURORA_KP_THRESHOLD),
                    alpha=0.15, color="#ef4444", label="Aurora events")
    ax.set_title(title, fontweight="bold"); ax.set_ylabel("Kp Index")
    ax.set_ylim(-0.2, 9.5); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.show()

print("Helper functions defined ✓")

Helper functions defined ✓


In [ ]:
t3 = time.time()
print("=" * 60)
print("  STAGE 4 — XGBOOST + PSO (1–3 day forecast)")
print("=" * 60)

feat_df = df[selected_features + ["date", TARGET_COL]].dropna().reset_index(drop=True)
n       = len(feat_df)
n_test  = int(n * TEST_SPLIT); n_val = int(n * VAL_SPLIT); n_train = n - n_test - n_val

X_all = feat_df[selected_features].values.astype(np.float32)
y_all = feat_df[TARGET_COL].values.astype(np.float32)

X_train, y_train = X_all[:n_train],              y_all[:n_train]
X_val,   y_val   = X_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val]
X_test,  y_test  = X_all[n_train+n_val:],        y_all[n_train+n_val:]
dates_test       = feat_df["date"].iloc[n_train+n_val:].values

print(f"  Train: {n_train:,}  Val: {n_val:,}  Test: {n_test:,}")

# PSO tuning
pso = PSOHyperparamOptimiser(XGB_SEARCH_SPACE)
best_xgb_params, pso_mae = pso.run(X_train, y_train)
pd.DataFrame(pso.history).to_csv(OUT_XGB / "pso_convergence.csv", index=False)

# Train final model on train+val
X_tv = np.vstack([X_train, X_val]); y_tv = np.concatenate([y_train, y_val])
xgb_model = xgb.XGBRegressor(**best_xgb_params, random_state=SEED,
                               verbosity=0, tree_method="hist")
xgb_model.fit(X_tv, y_tv)
joblib.dump(xgb_model, OUT_MODELS / "xgboost_model.pkl")

# Evaluate
y_pred_xgb = xgb_model.predict(X_test)
xgb_metrics = _compute_metrics("XGBoost", y_test, y_pred_xgb)
print(f"\n  XGBoost test metrics:"); _print_metrics(xgb_metrics)

# Save predictions
xgb_pred_df = pd.DataFrame({
    "date": dates_test, "actual_kp": y_test, "predicted_kp": y_pred_xgb,
    "aurora_actual": (y_test >= AURORA_KP_THRESHOLD).astype(int),
    "aurora_predicted": (y_pred_xgb >= AURORA_KP_THRESHOLD).astype(int),
})
xgb_pred_df.to_csv(OUT_XGB / "xgboost_test_predictions.csv", index=False)

# Feature importance
fi = (pd.DataFrame({"feature": selected_features,
                    "importance": xgb_model.feature_importances_})
        .sort_values("importance", ascending=False))
fi.to_csv(OUT_XGB / "xgboost_feature_importance.csv", index=False)

# PSO convergence plot
pso_hist = pd.DataFrame(pso.history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(pso_hist["iter"], pso_hist["best_fit"], "o-", color="#f59e0b", label="Best MAE")
ax1.plot(pso_hist["iter"], pso_hist["mean_fit"], "--",  color="#94a3b8", label="Mean MAE", alpha=0.7)
ax1.set_xlabel("Iteration"); ax1.set_ylabel("CV-MAE")
ax1.set_title("PSO Convergence — XGBoost"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.barh(fi.head(20)["feature"], fi.head(20)["importance"],
         color="#3b82f6", alpha=0.8)
ax2.set_xlabel("Importance"); ax2.set_title("XGBoost Top 20 Features")
ax2.invert_yaxis(); ax2.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_PLOTS / "xgb_pso_and_importance.png", dpi=150, bbox_inches="tight")
plt.show()

_plot_predictions(dates_test, y_test, y_pred_xgb,
                  "XGBoost — Test Set Predictions vs Actuals",
                  COLORS["XGBoost"], OUT_PLOTS / "xgb_predictions.png")

print(f"\n  ⏱  XGBoost + PSO: {(time.time()-t3)/60:.1f} min")

  STAGE 4 — XGBOOST + PSO (1–3 day forecast)
  Train: 24,092  Val: 5,162  Test: 5,162
    PSO subsampled to 5,000 rows
  PSO  (particles=10, iters=10)
    Iter   1: best MAE=0.0336
    Iter   2: best MAE=0.0332
    Iter   3: best MAE=0.0320
    Iter   4: best MAE=0.0297
    Iter   5: best MAE=0.0291
    Iter   6: best MAE=0.0291
    Iter   7: best MAE=0.0289
    Iter   8: best MAE=0.0288
    Iter   9: best MAE=0.0288
    Iter  10: best MAE=0.0288

  PSO best MAE : 0.0288
  Best params  : {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.7001418451793016, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'reg_lambda': 1.1870635457171421}

  XGBoost test metrics:
    RMSE=0.0323  MAE=0.0195  R²=0.9990  MAPE=567296.4%  |  Aurora Prec=0.971  Rec=1.000  F1=0.985

  ⏱  XGBoost + PSO: 3.2 min


## 7 — LSTM Models (medium-term forecast)

In [ ]:
def create_sequences(X_data, y_data, lookback):
    """Build (X, y) sequence pairs for LSTM input."""
    X, y = [], []
    for i in range(lookback, len(X_data)):
        X.append(X_data[i-lookback:i]); y.append(y_data[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def build_stacked_lstm(lookback, n_features):
    u = LSTM_UNITS
    model = Sequential([
        Input(shape=(lookback, n_features)),
        LSTM(u, return_sequences=True,  kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.30),
        LSTM(u, return_sequences=True,  kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.25),
        LSTM(u, return_sequences=True,  kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.20),
        LSTM(u, return_sequences=False),
        BatchNormalization(), Dropout(0.15),
        Dense(u, activation="relu", kernel_regularizer=l2(1e-4)), Dropout(0.10),
        Dense(u, activation="relu"), Dense(1, activation="sigmoid"),
    ], name="Stacked_LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="huber", metrics=["mae"])
    return model


def build_bilstm(lookback, n_features):
    u = LSTM_UNITS
    model = Sequential([
        Input(shape=(lookback, n_features)),
        Bidirectional(LSTM(u, return_sequences=True, kernel_regularizer=l2(1e-4))),
        BatchNormalization(), Dropout(0.30),
        Bidirectional(LSTM(u, return_sequences=True, kernel_regularizer=l2(1e-4))),
        BatchNormalization(), Dropout(0.25),
        Bidirectional(LSTM(u, return_sequences=False)),
        BatchNormalization(), Dropout(0.20),
        Dense(u, activation="relu"), Dropout(0.10),
        Dense(u, activation="relu"), Dense(1, activation="sigmoid"),
    ], name="BiLSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="huber", metrics=["mae"])
    return model


def build_cnn_lstm(lookback, n_features):
    u = LSTM_UNITS
    model = Sequential([
        Input(shape=(lookback, n_features)),
        Conv1D(u, kernel_size=3, activation="relu", padding="same", kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Conv1D(u, kernel_size=5, activation="relu", padding="same", kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling1D(2), Dropout(0.25),
        LSTM(u, return_sequences=True, kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.25),
        LSTM(u, return_sequences=False), BatchNormalization(), Dropout(0.20),
        Dense(u, activation="relu"), Dropout(0.10),
        Dense(u, activation="relu"), Dense(1, activation="sigmoid"),
    ], name="CNN_LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="huber", metrics=["mae"])
    return model


def build_attention_lstm(lookback, n_features):
    u      = LSTM_UNITS
    inputs = Input(shape=(lookback, n_features))
    x = LSTM(u, return_sequences=True, kernel_regularizer=l2(1e-4))(inputs)
    x = BatchNormalization()(x); x = Dropout(0.25)(x)
    x = LSTM(u, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    attn = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = Add()([x, attn]); x = LayerNormalization()(x); x = Dropout(0.20)(x)
    attn2 = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = Add()([x, attn2]); x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(u, activation="relu")(x); x = Dropout(0.15)(x)
    x = Dense(u, activation="relu")(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, out, name="Attn_LSTM")
    model.compile(optimizer=Adam(LEARNING_RATE), loss="huber", metrics=["mae"])
    return model


print("LSTM model builders defined ✓")
print(f"  LSTM units per layer: {LSTM_UNITS}")

LSTM model builders defined ✓
  LSTM units per layer: 48


In [ ]:
t4 = time.time()
print("=" * 60)
print("  STAGE 5 — LSTM MODELS (3–12 day forecast)")
print("=" * 60)

# Filter to LSTM_START_YEAR
lstm_df = df[df["date"].dt.year >= LSTM_START_YEAR].copy().reset_index(drop=True)
print(f"  LSTM data: {len(lstm_df):,} rows  "
      f"({lstm_df['date'].min().date()} → {lstm_df['date'].max().date()})")

feat_df_lstm = lstm_df[selected_features + ["date", TARGET_COL]].dropna().reset_index(drop=True)
nl = len(feat_df_lstm)
nl_test  = int(nl * TEST_SPLIT)
nl_val   = int(nl * VAL_SPLIT)
nl_train = nl - nl_test - nl_val

# Scale
feat_scaler   = RobustScaler()
target_scaler = MinMaxScaler()
X_all_l = feat_df_lstm[selected_features].values
y_all_l = feat_df_lstm[TARGET_COL].values.reshape(-1, 1)

X_tr_raw = feat_scaler.fit_transform(X_all_l[:nl_train])
y_tr_raw = target_scaler.fit_transform(y_all_l[:nl_train]).ravel()
X_va_raw = feat_scaler.transform(X_all_l[nl_train:nl_train+nl_val])
y_va_raw = target_scaler.transform(y_all_l[nl_train:nl_train+nl_val]).ravel()
X_te_raw = feat_scaler.transform(X_all_l[nl_train+nl_val:])
y_te_raw = target_scaler.transform(y_all_l[nl_train+nl_val:]).ravel()
joblib.dump(feat_scaler,   OUT_MODELS / "lstm_feat_scaler.pkl")
joblib.dump(target_scaler, OUT_MODELS / "lstm_target_scaler.pkl")

X_train_l, y_train_l = create_sequences(X_tr_raw, y_tr_raw, LOOKBACK)
X_val_l,   y_val_l   = create_sequences(X_va_raw, y_va_raw, LOOKBACK)
X_test_l,  y_test_l  = create_sequences(X_te_raw, y_te_raw, LOOKBACK)
n_features = X_train_l.shape[2]
dates_test_l = feat_df_lstm["date"].iloc[
    nl_train + nl_val + LOOKBACK : nl_train + nl_val + LOOKBACK + len(y_test_l)].values

print(f"  Train: {X_train_l.shape}  Val: {X_val_l.shape}  Test: {X_test_l.shape}")
print(f"  Features per timestep: {n_features}")

def get_callbacks(name):
    return [
        EarlyStopping(monitor="val_loss", patience=8,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=4, min_lr=1e-6, verbose=0),
        ModelCheckpoint(str(OUT_MODELS / f"lstm_{name}.keras"),
                        monitor="val_loss", save_best_only=True, verbose=0),
    ]

model_configs = [
    (build_stacked_lstm,   "Stacked LSTM"),
    (build_bilstm,         "BiLSTM"),
    (build_cnn_lstm,       "CNN-LSTM"),
    (build_attention_lstm, "Attn-LSTM"),
]

lstm_results   = {}
lstm_histories = {}

for build_fn, name in model_configs:
    print(f"\n  Training {name}...")
    model = build_fn(LOOKBACK, n_features)
    hist  = model.fit(X_train_l, y_train_l,
                      validation_data=(X_val_l, y_val_l),
                      epochs=EPOCHS, batch_size=BATCH_SIZE,
                      callbacks=get_callbacks(name.replace(" ","_")),
                      verbose=0)
    best_val = min(hist.history["val_loss"])
    print(f"    Best val loss: {best_val:.5f}")

    y_pred_s = model.predict(X_test_l, verbose=0).ravel()
    y_pred   = target_scaler.inverse_transform(y_pred_s.reshape(-1,1)).ravel()
    y_true   = target_scaler.inverse_transform(y_test_l.reshape(-1,1)).ravel()

    metrics = _compute_metrics(name, y_true, y_pred); _print_metrics(metrics)

    safe = name.replace(" ","_").lower()
    pd.DataFrame({"date": dates_test_l, "actual_kp": y_true, "predicted_kp": y_pred,
                  "aurora_actual":    (y_true >= AURORA_KP_THRESHOLD).astype(int),
                  "aurora_predicted": (y_pred >= AURORA_KP_THRESHOLD).astype(int),
                 }).to_csv(OUT_LSTM / f"{safe}_predictions.csv", index=False)

    _plot_predictions(dates_test_l, y_true, y_pred,
                      f"{name} — Test Set Predictions vs Actuals",
                      COLORS.get(name, "#6366f1"),
                      OUT_PLOTS / f"lstm_{safe}_predictions.png")

    lstm_results[name]   = {"metrics": metrics, "y_true": y_true,
                             "y_pred": y_pred, "dates": dates_test_l}
    lstm_histories[name] = hist

print(f"\n  ⏱  LSTM training: {(time.time()-t4)/60:.1f} min")

  STAGE 5 — LSTM MODELS (3–12 day forecast)
  LSTM data: 9,579 rows  (2000-01-01 → 2026-03-23)
  Train: (6677, 30, 31)  Val: (1406, 30, 31)  Test: (1406, 30, 31)
  Features per timestep: 31

  Training Stacked LSTM...
    Best val loss: 0.00469
    RMSE=0.9494  MAE=0.7314  R²=0.2368  MAPE=51.6%  |  Aurora Prec=0.000  Rec=0.000  F1=0.000

  Training BiLSTM...
    Best val loss: 0.00487
    RMSE=0.9086  MAE=0.7084  R²=0.3010  MAPE=52.0%  |  Aurora Prec=1.000  Rec=0.048  F1=0.091

  Training CNN-LSTM...
    Best val loss: 0.00544
    RMSE=0.9266  MAE=0.7160  R²=0.2731  MAPE=54.8%  |  Aurora Prec=1.000  Rec=0.048  F1=0.091

  Training Attn-LSTM...
    Best val loss: 0.00662
    RMSE=1.1962  MAE=0.8919  R²=-0.2116  MAPE=56.4%  |  Aurora Prec=0.000  Rec=0.000  F1=0.000

  ⏱  LSTM training: 12.1 min


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("LSTM Training & Validation Loss", fontsize=14, fontweight="bold")
for ax, (name, hist) in zip(axes.ravel(), lstm_histories.items()):
    ax.plot(hist.history["loss"],     lw=1.5, color="#3b82f6", label="Train")
    ax.plot(hist.history["val_loss"], lw=1.5, color="#ef4444", ls="--", label="Val")
    best = int(np.argmin(hist.history["val_loss"]))
    ax.axvline(best, color="gray", ls=":", lw=1)
    ax.set_title(name, fontweight="bold"); ax.set_xlabel("Epoch"); ax.set_ylabel("Huber Loss")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_PLOTS / "lstm_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 — Evaluation & Ensemble

In [ ]:
print("=" * 60)
print("  STAGE 6 — EVALUATION & ENSEMBLE")
print("=" * 60)

all_metrics = [xgb_metrics] + [r["metrics"] for r in lstm_results.values()]
summary_df  = pd.DataFrame(all_metrics).sort_values("r2", ascending=False)
summary_df.to_csv(OUT_EVAL / "model_comparison.csv", index=False)

print("\n  Model Comparison:")
print(summary_df.to_string(index=False))

# Comparison bar chart
metrics_plot = ["rmse", "mae", "r2", "aurora_f1"]
labels_plot  = ["RMSE ↓", "MAE ↓", "R² ↑", "Aurora F1 ↑"]
palette      = [COLORS.get(m, "#6366f1") for m in summary_df["model"]]
fig, axes    = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle("Model Comparison — Kp & Aurora Metrics", fontsize=14, fontweight="bold")
for ax, metric, label in zip(axes, metrics_plot, labels_plot):
    ax.bar(summary_df["model"], summary_df[metric], color=palette, alpha=0.85)
    ax.set_title(label, fontweight="bold")
    ax.set_xticklabels(summary_df["model"], rotation=30, ha="right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_PLOTS / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Ensemble 
best_lstm_name = summary_df[summary_df["model"] != "XGBoost"].iloc[0]["model"]
best_lstm_res  = lstm_results[best_lstm_name]

xgb_series  = xgb_pred_df.set_index("date")["predicted_kp"]
lstm_series  = pd.Series(best_lstm_res["y_pred"],
                          index=pd.to_datetime(best_lstm_res["dates"]))
common_dates = xgb_series.index.intersection(lstm_series.index)

if len(common_dates) > 0:
    xgb_r2   = xgb_metrics["r2"]
    lstm_r2  = best_lstm_res["metrics"]["r2"]
    total    = (xgb_r2 + lstm_r2) if (xgb_r2 + lstm_r2) > 0 else 1
    w_xgb    = xgb_r2 / total;  w_lstm = lstm_r2 / total
    ens_pred = (w_xgb * xgb_series.loc[common_dates].values +
                w_lstm * lstm_series.loc[common_dates].values)
    true_vals = xgb_pred_df.set_index("date").loc[common_dates, "actual_kp"].values
    ens_metrics = _compute_metrics("Ensemble", true_vals, ens_pred)
    all_metrics.append(ens_metrics)
    print(f"\n  Ensemble  (XGBoost w={w_xgb:.2f}  +  {best_lstm_name} w={w_lstm:.2f}):")
    _print_metrics(ens_metrics)

    pd.DataFrame({
        "date": common_dates, "actual_kp": true_vals, "ensemble_kp": ens_pred,
        "xgboost_kp": xgb_series.loc[common_dates].values,
        f"{best_lstm_name.replace(' ','_').lower()}_kp":
            lstm_series.loc[common_dates].values,
    }).to_csv(OUT_EVAL / "ensemble_predictions.csv", index=False)

    _plot_predictions(common_dates, true_vals, ens_pred,
                      "Ensemble — Test Set Predictions vs Actuals",
                      COLORS["Ensemble"],
                      OUT_PLOTS / "ensemble_predictions.png")

final_df = pd.DataFrame(all_metrics).sort_values("r2", ascending=False)
final_df.to_csv(OUT_EVAL / "final_model_summary.csv", index=False)

total_time = time.time() - t0
print(f"\n{'='*60}")
print(f"  PIPELINE COMPLETE  —  {total_time/60:.1f} min total")
print(f"{'='*60}")
print("\nOutput folder summary:")
for d in [OUT_CLEAN, OUT_FEAT, OUT_GA, OUT_XGB, OUT_LSTM, OUT_EVAL, OUT_MODELS, OUT_PLOTS]:
    files = list(d.glob("*"))
    print(f"  {d.name}/  ({len(files)} file{'s' if len(files)!=1 else ''})")
final_df

  STAGE 6 — EVALUATION & ENSEMBLE

  Model Comparison:
       model   rmse    mae      r2      mape  aurora_precision  aurora_recall  aurora_f1
     XGBoost 0.0323 0.0195  0.9990 567296.38             0.971          1.000      0.985
      BiLSTM 0.9086 0.7084  0.3010     51.98             1.000          0.048      0.091
    CNN-LSTM 0.9266 0.7160  0.2731     54.78             1.000          0.048      0.091
Stacked LSTM 0.9494 0.7314  0.2368     51.58             0.000          0.000      0.000
   Attn-LSTM 1.1962 0.8919 -0.2116     56.38             0.000          0.000      0.000

  Ensemble  (XGBoost w=0.77  +  BiLSTM w=0.23):
    RMSE=0.2132  MAE=0.1648  R²=0.9615  MAPE=12.1%  |  Aurora Prec=1.000  Rec=0.476  F1=0.645

  PIPELINE COMPLETE  —  17.3 min total

Output folder summary:
  01_cleaned_data/  (1 file)
  02_feature_data/  (1 file)
  03_ga_feature_selection/  (2 files)
  04_xgboost/  (3 files)
  05_lstm/  (4 files)
  06_evaluation/  (7 files)
  07_models/  (7 files)
  08_plot

,model,rmse,mae,r2,mape,aurora_precision,aurora_recall,aurora_f1
0,XGBoost,0.0323,0.0195,0.9990,567296.38,0.971,1.000,0.985
5,Ensemble,0.2132,0.1648,0.9615,12.12,1.000,0.476,0.645
2,BiLSTM,0.9086,0.7084,0.3010,51.98,1.000,0.048,0.091
3,CNN-LSTM,0.9266,0.7160,0.2731,54.78,1.000,0.048,0.091
1,Stacked LSTM,0.9494,0.7314,0.2368,51.58,0.000,0.000,0.000
4,Attn-LSTM,1.1962,0.8919,-0.2116,56.38,0.000,0.000,0.000


## 9 — Live NOAA Validation

In [ ]:


import urllib.request
import json as _json
from IPython.display import display

print('=' * 65)
print('  LIVE NOAA VALIDATION')
print('=' * 65)

OBS_URL  = 'https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json'
FCST_URL = 'https://services.swpc.noaa.gov/products/noaa-planetary-k-index-forecast.json'
GFZ_URL  = 'https://kp.gfz-potsdam.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt'

def _fetch_json(url):
    with urllib.request.urlopen(url, timeout=15) as r:
        return _json.loads(r.read().decode())

try:
    obs_raw  = _fetch_json(OBS_URL)
    fcst_raw = _fetch_json(FCST_URL)
    print(f'  Observed  : {len(obs_raw)-1} records')
    print(f'  Forecast  : {len(fcst_raw)-1} records')
    noaa_ok = True
except Exception as e:
    print(f'  WARNING: could not fetch NOAA data: {e}')
    noaa_ok = False

if noaa_ok:
    # Parse observed
    obs_rows = []
    for row in obs_raw[1:]:
        try:
            obs_rows.append({'timestamp': pd.to_datetime(row[0]),
                             'kp': float(row[1])})
        except Exception:
            continue
    obs_df = pd.DataFrame(obs_rows).dropna()
    obs_df['date'] = obs_df['timestamp'].dt.floor('D')
    obs_daily = (obs_df.groupby('date')['kp'].mean().reset_index()
                   .rename(columns={'kp': 'kp_noaa_observed'}))
    obs_daily['date'] = pd.to_datetime(obs_daily['date'])

    # Parse NOAA forecast
    fcst_rows = []
    for row in fcst_raw[1:]:
        try:
            fcst_rows.append({'timestamp': pd.to_datetime(row[0]),
                              'kp': float(row[1]),
                              'type': str(row[2]).strip().lower()})
        except Exception:
            continue
    fcst_df = pd.DataFrame(fcst_rows).dropna()
    fcst_df['date'] = fcst_df['timestamp'].dt.floor('D')
    noaa_pred_daily = (fcst_df[fcst_df['type'] == 'predicted']
                         .groupby('date')['kp'].mean().reset_index()
                         .rename(columns={'kp': 'kp_noaa_forecast'}))
    noaa_pred_daily['date'] = pd.to_datetime(noaa_pred_daily['date'])

    live_window_start = obs_daily['date'].min()
    live_window_end   = obs_daily['date'].max()
    training_end      = df['date'].max()
    gap_days          = (live_window_start - training_end).days

    print(f'  Training ends : {training_end.date()}')
    print(f'  NOAA window   : {live_window_start.date()} → {live_window_end.date()}')
    print(f'  Gap           : {gap_days} days')


if noaa_ok and gap_days > 0:
    print(f'\n  Gap detected — downloading latest GFZ Kp data to patch...')
    try:
        gfz_path = TRAIN_DIR / 'kp_latest_auto.txt'
        urllib.request.urlretrieve(GFZ_URL, str(gfz_path))
        df_new = parse_kp_file(gfz_path)
        print(f'  GFZ download: {len(df_new):,} rows → ends {df_new["date"].max().date()}')

        # Engineer features for new rows only, then merge
        df_new_eng = engineer_features(df_new)
        df = (pd.concat([df, df_new_eng])
                .drop_duplicates('date')
                .sort_values('date')
                .reset_index(drop=True))
        print(f'  df updated: {len(df):,} rows, ends {df["date"].max().date()}')
        gap_days = (live_window_start - df['date'].max()).days
        print(f'  Remaining gap: {max(0, gap_days)} days')
    except Exception as e:
        print(f'  WARNING: GFZ download failed: {e}')
        print('  Falling back to test-set evaluation.')


if noaa_ok:
    live_feat    = df[df['date'].isin(obs_daily['date'].values)]
    live_feat    = live_feat[selected_features + ['date', TARGET_COL]].dropna()
    overlap_days = len(live_feat)
else:
    overlap_days = 0

USE_LIVE = overlap_days >= 3
print(f'\n  Mode: {"LIVE" if USE_LIVE else "TEST-SET FALLBACK"} ({overlap_days} overlap days)')


best_lstm_name_live = max(lstm_results,
                           key=lambda n: lstm_results[n]['metrics']['r2'])
safe_lstm = best_lstm_name_live.replace(' ', '_')

if USE_LIVE:
    combined = obs_daily.copy()
    combined = combined.merge(noaa_pred_daily, on='date', how='left')

    # XGBoost
    X_live = live_feat[selected_features].values.astype(np.float32)
    xgb_preds = xgb_model.predict(X_live)
    combined  = combined.merge(
        pd.DataFrame({'date': live_feat['date'].values,
                      'kp_xgboost': xgb_preds}),
        on='date', how='left')
    print(f'  XGBoost: {len(live_feat)} predictions')

    # Best LSTM
    lstm_path = OUT_MODELS / f'lstm_{safe_lstm}.keras'
    if lstm_path.exists() and len(live_feat) >= LOOKBACK:
        from tensorflow.keras.models import load_model as _lm
        _lstm   = _lm(str(lstm_path))
        idx0    = df[df['date'] == live_feat['date'].iloc[0]].index[0]
        start_i = max(0, idx0 - LOOKBACK)
        seq_df  = df.iloc[start_i : idx0 + len(live_feat)]
        seq_sc  = feat_scaler.transform(seq_df[selected_features].values)
        X_seq, _ = create_sequences(seq_sc, np.zeros(len(seq_sc)), LOOKBACK)
        if len(X_seq) > 0:
            preds  = target_scaler.inverse_transform(
                         _lstm.predict(X_seq, verbose=0)).ravel()
            sdates = seq_df['date'].iloc[LOOKBACK:].values
            sdates = sdates[sdates >= live_feat['date'].iloc[0]]
            trim   = len(sdates)
            combined = combined.merge(
                pd.DataFrame({'date': sdates,
                              f'kp_{safe_lstm.lower()}': preds[-trim:]}),
                on='date', how='left')
            print(f'  {best_lstm_name_live}: {trim} predictions')

    ground_truth_col = 'kp_noaa_observed'
    source_label     = 'NOAA Live Observed'

else:
    # Test-set fallback — load saved predictions
    print('  Loading saved test-set predictions...')
    xgb_test = pd.read_csv(OUT_XGB / 'xgboost_test_predictions.csv',
                            parse_dates=['date'])
    combined  = xgb_test[['date','actual_kp','predicted_kp']].rename(
        columns={'actual_kp': 'kp_noaa_observed',
                 'predicted_kp': 'kp_xgboost'})

    lstm_csv = OUT_LSTM / f'{safe_lstm.lower()}_predictions.csv'
    if lstm_csv.exists():
        lt = pd.read_csv(lstm_csv, parse_dates=['date'])
        combined = combined.merge(
            lt[['date','predicted_kp']].rename(
                columns={'predicted_kp': f'kp_{safe_lstm.lower()}'}),
            on='date', how='left')

    combined['kp_noaa_forecast'] = np.nan
    ground_truth_col = 'kp_noaa_observed'
    source_label     = 'Historical Test Set (GFZ Kp)'
    print(f'  Test rows: {len(combined)}  '
          f'({combined["date"].min().date()} → {combined["date"].max().date()})')

combined = combined.dropna(subset=[ground_truth_col]).reset_index(drop=True)
combined.to_csv(OUT_EVAL / 'live_noaa_validation.csv', index=False)


print('\n' + '=' * 65)
print(f'  ACCURACY vs {source_label}')
print('=' * 65)

prediction_cols = {}
if 'kp_noaa_forecast' in combined and combined['kp_noaa_forecast'].notna().sum() > 1:
    prediction_cols['NOAA Official Forecast'] = 'kp_noaa_forecast'
if 'kp_xgboost' in combined.columns:
    prediction_cols['XGBoost (our model)'] = 'kp_xgboost'
lstm_col = f'kp_{safe_lstm.lower()}'
if lstm_col in combined.columns:
    prediction_cols[f'{best_lstm_name_live} (our model)'] = lstm_col

accuracy_rows = []
y_obs = combined[ground_truth_col].values

for label, col in prediction_cols.items():
    mask = combined[col].notna()
    y_t  = y_obs[mask]
    y_p  = combined.loc[mask, col].values
    if len(y_t) < 2:
        continue
    rmse      = float(np.sqrt(mean_squared_error(y_t, y_p)))
    mae       = float(mean_absolute_error(y_t, y_p))
    r2        = float(r2_score(y_t, y_p))
    within1   = float(np.mean(np.abs(y_t - y_p) <= 1.0) * 100)
    within05  = float(np.mean(np.abs(y_t - y_p) <= 0.5) * 100)
    t_au      = (y_t >= AURORA_KP_THRESHOLD).astype(int)
    p_au      = (y_p >= AURORA_KP_THRESHOLD).astype(int)
    aurora_acc = float(np.mean(t_au == p_au) * 100)
    row = {'Model': label, 'N days': int(mask.sum()),
           'RMSE': round(rmse,3), 'MAE': round(mae,3), 'R²': round(r2,3),
           'Within ±1.0 Kp %': round(within1,1),
           'Within ±0.5 Kp %': round(within05,1),
           'Aurora Event Acc %': round(aurora_acc,1)}
    accuracy_rows.append(row)
    print(f"\n  {label}")
    for k, v in row.items():
        if k != 'Model':
            print(f"    {k:<22}: {v}")

acc_df = pd.DataFrame(accuracy_rows)
acc_df.to_csv(OUT_EVAL / 'live_accuracy_summary.csv', index=False)


plt.rcParams.update({'figure.dpi': 130, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False,
                     'axes.spines.right': False})

PLOT_COLORS = {
    'NOAA Official Forecast':       '#64748b',
    'XGBoost (our model)':          '#f59e0b',
    'Stacked LSTM (our model)':     '#3b82f6',
    'BiLSTM (our model)':           '#10b981',
    'CNN-LSTM (our model)':         '#f59e0b',
    'Attn-LSTM (our model)':        '#8b5cf6',
}

fig = plt.figure(figsize=(18, 16))
gs  = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.35)
dates_plot = pd.to_datetime(combined['date'])

# Panel 1: time series
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(dates_plot, combined[ground_truth_col], lw=2.0,
         color='#1e293b', label=f'Observed Kp ({source_label})', zorder=5)
ax1.axhline(AURORA_KP_THRESHOLD, color='#dc2626', lw=0.8, ls=':', alpha=0.7)
ax1.fill_between(dates_plot, AURORA_KP_THRESHOLD, 9.5,
                 where=(combined[ground_truth_col] >= AURORA_KP_THRESHOLD),
                 alpha=0.12, color='#ef4444', label='Aurora events')
for label, col in prediction_cols.items():
    if col in combined.columns and combined[col].notna().sum() > 0:
        ax1.plot(dates_plot, combined[col], lw=1.4, ls='--',
                 color=PLOT_COLORS.get(label, '#6366f1'),
                 label=label, alpha=0.85)
ax1.set_title(f'Kp Predictions vs {source_label}',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('Kp Index'); ax1.set_ylim(-0.3, 9.5)
ax1.legend(fontsize=9, loc='upper left')
fmt = mdates.DateFormatter('%b %d') if USE_LIVE else mdates.DateFormatter('%Y-%m')
ax1.xaxis.set_major_formatter(fmt)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=20)

# Panels 2 & 3: scatter plots
for (r, c), (label, col) in zip([(1,0),(1,1)],
                                  list(prediction_cols.items())[:2]):
    ax = fig.add_subplot(gs[r, c])
    if col not in combined.columns:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='gray')
        ax.set_title(label, fontsize=10, fontweight='bold'); continue
    mask = combined[col].notna()
    y_t  = combined.loc[mask, ground_truth_col].values
    y_p  = combined.loc[mask, col].values
    if len(y_t) < 2:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='gray')
        ax.set_title(label, fontsize=10, fontweight='bold'); continue
    ax.scatter(y_t, y_p, color=PLOT_COLORS.get(label, '#6366f1'),
               alpha=0.7, s=40, edgecolors='white', linewidths=0.5)
    lims = [min(y_t.min(), y_p.min()) - 0.3,
            max(y_t.max(), y_p.max()) + 0.3]
    ax.plot(lims, lims, 'k--', lw=1, alpha=0.4, label='Perfect forecast')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Observed Kp'); ax.set_ylabel('Predicted Kp')
    ax.set_title(f'{label}\nR²={r2_score(y_t, y_p):.3f}',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

# Panel 4: accuracy bar chart
if not acc_df.empty:
    ax4 = fig.add_subplot(gs[2, 0])
    x   = np.arange(len(acc_df))
    w   = 0.25
    b1  = ax4.bar(x-w, acc_df['Within ±1.0 Kp %'], w,
                  label='Within ±1.0 Kp', color='#3b82f6', alpha=0.85)
    b2  = ax4.bar(x,   acc_df['Within ±0.5 Kp %'], w,
                  label='Within ±0.5 Kp', color='#10b981', alpha=0.85)
    b3  = ax4.bar(x+w, acc_df['Aurora Event Acc %'], w,
                  label='Aurora Event Acc', color='#ef4444', alpha=0.85)
    ax4.set_xticks(x)
    ax4.set_xticklabels(
        [m.replace(' (our model)', '') for m in acc_df['Model']],
        rotation=15, ha='right', fontsize=9)
    ax4.set_ylabel('Accuracy (%)'); ax4.set_ylim(0, 110)
    ax4.set_title('Prediction Accuracy', fontsize=11, fontweight='bold')
    ax4.legend(fontsize=8)
    for bar in list(b1) + list(b2) + list(b3):
        h = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2, h + 1,
                 f'{h:.0f}%', ha='center', va='bottom', fontsize=7)

# Panel 5: error distribution
ax5 = fig.add_subplot(gs[2, 1])
for label, col in prediction_cols.items():
    if col not in combined.columns: continue
    mask   = combined[col].notna()
    errors = (combined.loc[mask, col] -
              combined.loc[mask, ground_truth_col]).values
    if len(errors) > 2:
        ax5.hist(errors, bins=20, alpha=0.55,
                 color=PLOT_COLORS.get(label, '#6366f1'),
                 label=label.replace(' (our model)', ''),
                 edgecolor='white')
ax5.axvline(0, color='black', lw=1.2, ls='--', alpha=0.6)
ax5.set_xlabel('Prediction Error (Predicted − Observed Kp)')
ax5.set_ylabel('Count')
ax5.set_title('Error Distribution', fontsize=11, fontweight='bold')
ax5.legend(fontsize=8)

plt.savefig(OUT_PLOTS / 'live_noaa_validation.png',
            dpi=150, bbox_inches='tight')
plt.show()


print('\n' + '=' * 65)
print('  FINAL ACCURACY SUMMARY')
print('=' * 65)

if acc_df.empty:
    print('\n  No metrics computed — no valid prediction/observation pairs.')
    print('  Training data may still not reach the live window.')
    print(f'  Training ends : {df["date"].max().date()}')
    if noaa_ok:
        print(f'  NOAA window   : {obs_daily["date"].min().date()}')
        print('  Manual fix    : ' + GFZ_URL)
else:
    display(acc_df.set_index('Model'))
    our_models = acc_df[acc_df['Model'].str.contains('our model')]
    noaa_row   = acc_df[acc_df['Model'] == 'NOAA Official Forecast']
    if not our_models.empty:
        best = our_models.loc[our_models['Within ±1.0 Kp %'].idxmax()]
        print(f'\n  Best model      : {best["Model"]}')
        print(f'  Within ±1.0 Kp  : {best["Within ±1.0 Kp %"]:.1f}%')
        print(f'  Aurora event acc: {best["Aurora Event Acc %"]:.1f}%')
        if not noaa_row.empty:
            delta     = best['Within ±1.0 Kp %'] - noaa_row.iloc[0]['Within ±1.0 Kp %']
            direction = 'BETTER' if delta >= 0 else 'WORSE'
            print(f'  vs NOAA forecast: {delta:+.1f} pp ({direction})')

if not USE_LIVE:
    print('\n  NOTE: Results on historical test set — not live NOAA data.')
    print('  Live comparison requires training data to reach current date.')
    print('  Auto-download attempted from GFZ; re-run this cell to retry.')

print(f'\n  Saved → {OUT_EVAL}/live_noaa_validation.csv')
print(f'  Saved → {OUT_EVAL}/live_accuracy_summary.csv')
print(f'  Saved → {OUT_PLOTS}/live_noaa_validation.png')


  LIVE NOAA VALIDATION
  Observed  : 59 records
  Forecast  : 81 records
  Training ends : 2026-03-23
  NOAA window   : 2026-03-18 → 2026-03-25
  Gap           : -5 days

  Mode: LIVE (6 overlap days)
  XGBoost: 6 predictions

  ACCURACY vs NOAA Live Observed

  XGBoost (our model)
    N days                : 6
    RMSE                  : 0.23
    MAE                   : 0.198
    R²                    : 0.983
    Within ±1.0 Kp %      : 100.0
    Within ±0.5 Kp %      : 100.0
    Aurora Event Acc %    : 100.0

  FINAL ACCURACY SUMMARY


,N days,RMSE,MAE,R²,Within ±1.0 Kp %,Within ±0.5 Kp %,Aurora Event Acc %
Model,,,,,,,
XGBoost (our model),6,0.23,0.198,0.983,100.0,100.0,100.0



  Best model      : XGBoost (our model)
  Within ±1.0 Kp  : 100.0%
  Aurora event acc: 100.0%

  Saved → /content/drive/MyDrive/data/output-data/06_evaluation/live_noaa_validation.csv
  Saved → /content/drive/MyDrive/data/output-data/06_evaluation/live_accuracy_summary.csv
  Saved → /content/drive/MyDrive/data/output-data/08_plots/live_noaa_validation.png


In [ ]:


import urllib.request, json as _json
from IPython.display import display

print('=' * 65)
print('  FULL LIVE NOAA COMPARISON')
print('=' * 65)
URLS = {
    'mag':     'https://services.swpc.noaa.gov/products/solar-wind/mag-7-day.json',
    'plasma':  'https://services.swpc.noaa.gov/products/solar-wind/plasma-7-day.json',
    'kp_obs':  'https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json',
    'kp_fcst': 'https://services.swpc.noaa.gov/products/noaa-planetary-k-index-forecast.json',
    'dst':     'https://services.swpc.noaa.gov/products/kyoto-dst.json',
    'f107':    'https://services.swpc.noaa.gov/json/f107_cm_flux.json',
}
GFZ_URL = 'https://kp.gfz-potsdam.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt'

def _fetch(url):
    with urllib.request.urlopen(url, timeout=20) as r:
        return _json.loads(r.read().decode())


print('\n[1/5] Fetching live NOAA data...')
raw = {}
for name, url in URLS.items():
    try:
        raw[name] = _fetch(url)
        print(f'      {name:<10}: {len(raw[name])-1} records')
    except Exception as e:
        print(f'      {name:<10}: FAILED ({e})')
        raw[name] = []


print('\n[2/5] Parsing into daily averages...')

def _to_daily(rows, ts_col, val_col, out_col, fill_val=None):
    """Parse a list-of-lists JSON into a daily mean Series."""
    data = []
    for row in rows:
        try:
            ts  = pd.to_datetime(row[ts_col])
            val = float(row[val_col])
            if fill_val is not None and val == fill_val:
                continue
            data.append({'date': ts.floor('D'), 'v': val})
        except Exception:
            continue
    if not data:
        return pd.DataFrame(columns=['date', out_col])
    df_ = pd.DataFrame(data)
    return (df_.groupby('date')['v'].mean().reset_index()
              .rename(columns={'v': out_col})
              .assign(date=lambda x: pd.to_datetime(x['date'])))

# MAG — header: [time_tag, bx_gsm, by_gsm, bz_gsm, lon_gsm, lat_gsm, bt]
mag_rows = raw['mag'][1:] if len(raw['mag']) > 1 else []
mag_daily = _to_daily(mag_rows, 0, 3, 'Bz_GSM', fill_val=-999.9)   # col 3 = bz_gsm
bx_daily  = _to_daily(mag_rows, 0, 1, 'Bx',     fill_val=-999.9)
by_daily  = _to_daily(mag_rows, 0, 2, 'By_GSE',  fill_val=-999.9)
bt_daily  = _to_daily(mag_rows, 0, 6, 'B_mag',   fill_val=-999.9)

# Plasma — header: [time_tag, density, speed, temperature]
pla_rows   = raw['plasma'][1:] if len(raw['plasma']) > 1 else []
spd_daily  = _to_daily(pla_rows, 0, 2, 'sw_speed',   fill_val=-9999.0)
den_daily  = _to_daily(pla_rows, 0, 1, 'sw_density',  fill_val=-9999.0)
tmp_daily  = _to_daily(pla_rows, 0, 3, 'sw_temp',     fill_val=-1.0e10)

# Kp observed — header: [time_tag, kp_index, observed, noaa_scale]
kp_obs_rows  = raw['kp_obs'][1:]  if len(raw['kp_obs'])  > 1 else []
kp_daily_obs = _to_daily(kp_obs_rows, 0, 1, 'kp_noaa_observed')

# Kp forecast
kp_fcst_rows = raw['kp_fcst'][1:] if len(raw['kp_fcst']) > 1 else []
fcst_pred = [r for r in kp_fcst_rows
             if len(r) > 2 and str(r[2]).strip().lower() == 'predicted']
kp_daily_fcst = _to_daily(fcst_pred, 0, 1, 'kp_noaa_forecast')

# Dst — [[time_tag, dst], ...]
dst_rows = []
for item in (raw['dst'] if isinstance(raw['dst'], list) else []):
    try:
        if isinstance(item, list):
            dst_rows.append({'date': pd.to_datetime(item[0]).floor('D'),
                             'v': float(item[1])})
        elif isinstance(item, dict):
            ts_k = next((k for k in item if 'time' in k.lower()), None)
            v_k  = next((k for k in item if k != ts_k), None)
            if ts_k and v_k:
                dst_rows.append({'date': pd.to_datetime(item[ts_k]).floor('D'),
                                 'v': float(item[v_k])})
    except Exception:
        continue
dst_daily = (pd.DataFrame(dst_rows).groupby('date')['v'].mean().reset_index()
               .rename(columns={'v': 'swpc_dst'})
               .assign(date=lambda x: pd.to_datetime(x['date']))
             if dst_rows else pd.DataFrame(columns=['date','swpc_dst']))

# F10.7 — [[time_tag, flux], ...]
f107_rows = []
for item in (raw['f107'] if isinstance(raw['f107'], list) else []):
    try:
        if isinstance(item, list):
            f107_rows.append({'date': pd.to_datetime(item[0]).floor('D'),
                              'v': float(item[1])})
        elif isinstance(item, dict):
            ts_k = next((k for k in item if 'time' in k.lower()), None)
            v_k  = next((k for k in item if 'flux' in k.lower() or 'f107' in k.lower()), None)
            if not v_k:
                v_k = next((k for k in item if k != ts_k), None)
            if ts_k and v_k:
                f107_rows.append({'date': pd.to_datetime(item[ts_k]).floor('D'),
                                  'v': float(item[v_k])})
    except Exception:
        continue
f107_daily = (pd.DataFrame(f107_rows).groupby('date')['v'].mean().reset_index()
                .rename(columns={'v': 'F107obs'})
                .assign(date=lambda x: pd.to_datetime(x['date']))
              if f107_rows else pd.DataFrame(columns=['date','F107obs']))

print(f'      MAG daily rows   : {len(mag_daily)}')
print(f'      Plasma daily rows: {len(spd_daily)}')
print(f'      Kp observed days : {len(kp_daily_obs)}')
print(f'      Kp forecast days : {len(kp_daily_fcst)}')
print(f'      Dst daily rows   : {len(dst_daily)}')
print(f'      F10.7 daily rows : {len(f107_daily)}')


print('\n[3/5] Downloading GFZ for Kp lag features...')
gfz_path = TRAIN_DIR / 'kp_live_full.txt'
urllib.request.urlretrieve(GFZ_URL, str(gfz_path))
df_gfz = parse_kp_file(gfz_path)
df_gfz_eng = engineer_features(df_gfz)
print(f'      GFZ ends: {df_gfz_eng["date"].max().date()}')

# Merge all live solar wind sources into df_gfz_eng
for src in [mag_daily, bx_daily, by_daily, bt_daily,
            spd_daily, den_daily, tmp_daily, dst_daily, f107_daily]:
    if len(src) > 0:
        src['date'] = pd.to_datetime(src['date'])
        df_gfz_eng = df_gfz_eng.merge(src, on='date', how='left',
                                        suffixes=('', '_live'))
        # Prefer live values over GFZ-derived where available
        for col in src.columns:
            if col == 'date':
                continue
            live_col = col + '_live'
            if live_col in df_gfz_eng.columns:
                df_gfz_eng[col] = df_gfz_eng[live_col].combine_first(
                                      df_gfz_eng.get(col, np.nan))
                df_gfz_eng.drop(columns=[live_col], inplace=True)

# Re-compute derived features that depend on solar wind
if 'Bz_GSM' in df_gfz_eng.columns:
    df_gfz_eng['Bz_min_3d']  = df_gfz_eng['Bz_GSM'].rolling(3, min_periods=1).min()
    df_gfz_eng['Bz_mean_7d'] = df_gfz_eng['Bz_GSM'].rolling(7, min_periods=1).mean()
    for lag in [1,2,3,7]:
        df_gfz_eng[f'Bz_lag{lag}'] = df_gfz_eng['Bz_GSM'].shift(lag)
if 'sw_speed' in df_gfz_eng.columns:
    df_gfz_eng['speed_mean_7d'] = df_gfz_eng['sw_speed'].rolling(7, min_periods=1).mean()
    for lag in [1,2,3]:
        df_gfz_eng[f'speed_lag{lag}'] = df_gfz_eng['sw_speed'].shift(lag)
if 'By_GSE' in df_gfz_eng.columns:
    df_gfz_eng['By_GSM'] = df_gfz_eng['By_GSE']   # GSM ≈ GSE for daily averages

df_gfz_eng = df_gfz_eng.ffill().bfill()


print('\n[4/5] Running trained models on live dates...')

# All dates = observed + forecast union
all_dates = pd.concat([kp_daily_obs[['date']],
                        kp_daily_fcst[['date']]]).drop_duplicates().sort_values('date')
all_dates['date'] = pd.to_datetime(all_dates['date'])

# Check which selected_features are now available
available = [f for f in selected_features if f in df_gfz_eng.columns]
missing   = [f for f in selected_features if f not in df_gfz_eng.columns]
if missing:
    print(f'      Still missing {len(missing)} features (will use col mean):')
    print(f'      {missing}')

feat_rows = df_gfz_eng[df_gfz_eng['date'].isin(all_dates['date'].values)].copy()
feat_rows = feat_rows.reindex(columns=selected_features + ['date'])
# Fill any remaining gaps with training column means
train_means = df[selected_features].mean()
feat_rows[selected_features] = feat_rows[selected_features].fillna(train_means)
feat_rows = feat_rows.dropna(subset=['date']).reset_index(drop=True)

print(f'      Target dates     : {len(all_dates)}')
print(f'      Feature rows matched: {len(feat_rows)}')

if len(feat_rows) == 0:
    print('\n  ERROR: No feature rows available. GFZ may not yet cover the NOAA window.')
    print(f'  GFZ ends: {df_gfz_eng["date"].max().date()}')
    raise ValueError('Re-run tomorrow when GFZ updates.')

# XGBoost
X_live = feat_rows[selected_features].values.astype(np.float32)
xgb_live_preds = xgb_model.predict(X_live)
preds_df = pd.DataFrame({'date': feat_rows['date'].values,
                          'kp_xgboost': xgb_live_preds})
print(f'      XGBoost: {len(preds_df)} predictions')

# Best LSTM
best_lstm_name_live = max(lstm_results,
                           key=lambda n: lstm_results[n]['metrics']['r2'])
safe      = best_lstm_name_live.replace(' ', '_')
lstm_path = OUT_MODELS / f'lstm_{safe}.keras'

if lstm_path.exists():
    from tensorflow.keras.models import load_model as _lm
    _lstm_live = _lm(str(lstm_path))
    X_full_gfz = df_gfz_eng.reindex(columns=selected_features).fillna(train_means)
    seq_sc     = feat_scaler.transform(X_full_gfz.values)
    X_seq, _   = create_sequences(seq_sc, np.zeros(len(seq_sc)), LOOKBACK)
    seq_dates  = df_gfz_eng['date'].iloc[LOOKBACK:].values
    target_set = set(pd.to_datetime(feat_rows['date']).dt.date)
    idx_mask   = np.array([pd.Timestamp(d).date() in target_set
                            for d in seq_dates])
    if idx_mask.sum() > 0:
        lstm_sc = _lstm_live.predict(X_seq[idx_mask], verbose=0).ravel()
        lstm_pr = target_scaler.inverse_transform(
                      lstm_sc.reshape(-1,1)).ravel()
        lstm_df = pd.DataFrame({
            'date': pd.to_datetime(seq_dates[idx_mask]),
            f'kp_{safe.lower()}': lstm_pr})
        preds_df = preds_df.merge(lstm_df, on='date', how='left')
        print(f'      {best_lstm_name_live}: {idx_mask.sum()} predictions')
    else:
        print(f'      {best_lstm_name_live}: no sequence overlap')
else:
    print(f'      {best_lstm_name_live}: model file not found')


print('\n[5/5] Computing metrics and plotting...')

lstm_col = f'kp_{safe.lower()}'
combined = (all_dates
    .merge(kp_daily_obs,  on='date', how='left')
    .merge(kp_daily_fcst, on='date', how='left')
    .merge(preds_df,      on='date', how='left'))
combined.to_csv(OUT_EVAL / 'live_comparison_full.csv', index=False)

print('\nComparison table:')
print(combined.to_string(index=False))

# Metrics vs NOAA observed
print('\n' + '=' * 65)
print('  ACCURACY vs NOAA LIVE OBSERVED Kp')
print('=' * 65)

pred_cols = {'NOAA Official Forecast': 'kp_noaa_forecast',
             'XGBoost (our model)':    'kp_xgboost'}
if lstm_col in combined.columns:
    pred_cols[f'{best_lstm_name_live} (our model)'] = lstm_col

PLOT_COLORS = {
    'NOAA Official Forecast':       '#64748b',
    'XGBoost (our model)':          '#f59e0b',
    'Stacked LSTM (our model)':     '#3b82f6',
    'BiLSTM (our model)':           '#10b981',
    'CNN-LSTM (our model)':         '#f59e0b',
    'Attn-LSTM (our model)':        '#8b5cf6',
}

accuracy_rows = []
y_obs_all = combined['kp_noaa_observed'].values

for label, col in pred_cols.items():
    if col not in combined.columns:
        continue
    sub = combined[combined['kp_noaa_observed'].notna() & combined[col].notna()]
    if len(sub) < 2:
        print(f'\n  {label}: only {len(sub)} observed overlap days — skipping')
        continue
    y_t = sub['kp_noaa_observed'].values
    y_p = sub[col].values
    row = {
        'Model':             label,
        'N days':            len(sub),
        'RMSE':              round(float(np.sqrt(mean_squared_error(y_t,y_p))),3),
        'MAE':               round(float(mean_absolute_error(y_t,y_p)),3),
        'R²':                round(float(r2_score(y_t,y_p)),3),
        'Within ±1.0 Kp %':  round(float(np.mean(np.abs(y_t-y_p)<=1.0)*100),1),
        'Within ±0.5 Kp %':  round(float(np.mean(np.abs(y_t-y_p)<=0.5)*100),1),
        'Aurora Acc %':      round(float(np.mean(
                                 (y_t>=AURORA_KP_THRESHOLD).astype(int)==
                                 (y_p>=AURORA_KP_THRESHOLD).astype(int))*100),1),
    }
    accuracy_rows.append(row)
    print(f'\n  {label}  (n={len(sub)} days)')
    for k,v in row.items():
        if k not in ('Model','N days'):
            print(f'    {k:<20}: {v}')

acc_df_live = pd.DataFrame(accuracy_rows)
acc_df_live.to_csv(OUT_EVAL / 'live_accuracy_summary.csv', index=False)


plt.rcParams.update({'figure.dpi':130,'axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False})

fig, axes = plt.subplots(2, 2, figsize=(16,11))
fig.suptitle('Live NOAA Kp — Full Solar Wind Features vs Observed',
             fontsize=14, fontweight='bold')

dates_all = pd.to_datetime(combined['date'])
obs_mask  = combined['kp_noaa_observed'].notna()
fcst_mask = combined['kp_noaa_forecast'].notna()

# Panel 1: full window
ax = axes[0,0]
ax.plot(dates_all[obs_mask], combined.loc[obs_mask,'kp_noaa_observed'],
        lw=2.2, color='#1e293b', label='NOAA Observed',
        zorder=5, marker='o', ms=5)
if fcst_mask.sum() > 0:
    ax.plot(dates_all[fcst_mask], combined.loc[fcst_mask,'kp_noaa_forecast'],
            lw=1.6, color='#64748b', ls='--', label='NOAA Forecast', alpha=0.85)
for label, col in pred_cols.items():
    if col in combined.columns and combined[col].notna().sum() > 0:
        m = combined[col].notna()
        ax.plot(dates_all[m], combined.loc[m,col], lw=1.5, ls=':',
                color=PLOT_COLORS.get(label,'#6366f1'),
                label=label, alpha=0.9, marker='s', ms=4)
ax.axhline(AURORA_KP_THRESHOLD, color='#dc2626', lw=0.8, ls=':', alpha=0.6)
ax.fill_between(dates_all, AURORA_KP_THRESHOLD, 9.5,
                where=combined['kp_noaa_observed'].fillna(0)>=AURORA_KP_THRESHOLD,
                alpha=0.12, color='#ef4444', label='Aurora')
ax.set_title('Observed + Forecast Window', fontweight='bold')
ax.set_ylabel('Kp Index'); ax.set_ylim(-0.3,9.5)
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20)

# Panel 2: observed period zoomed
ax = axes[0,1]
obs_sub = combined[obs_mask]
d2 = pd.to_datetime(obs_sub['date'])
ax.plot(d2, obs_sub['kp_noaa_observed'], lw=2.2, color='#1e293b',
        label='NOAA Observed', marker='o', ms=5, zorder=5)
for label, col in pred_cols.items():
    if col in obs_sub.columns and obs_sub[col].notna().sum() > 0:
        ax.plot(d2, obs_sub[col], lw=1.5, ls='--',
                color=PLOT_COLORS.get(label,'#6366f1'),
                label=label, marker='s', ms=4, alpha=0.85)
ax.axhline(AURORA_KP_THRESHOLD, color='#dc2626', lw=0.8, ls=':', alpha=0.6)
ax.set_title('Observed Period — Model vs Truth', fontweight='bold')
ax.set_ylabel('Kp Index'); ax.set_ylim(-0.3,9.5)
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20)

# Panel 3: accuracy bars
ax = axes[1,0]
if not acc_df_live.empty:
    x = np.arange(len(acc_df_live)); w = 0.28
    b1 = ax.bar(x-w, acc_df_live['Within ±1.0 Kp %'], w,
                label='Within ±1.0 Kp', color='#3b82f6', alpha=0.85)
    b2 = ax.bar(x,   acc_df_live['Within ±0.5 Kp %'], w,
                label='Within ±0.5 Kp', color='#10b981', alpha=0.85)
    b3 = ax.bar(x+w, acc_df_live['Aurora Acc %'], w,
                label='Aurora Event Acc', color='#ef4444', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' (our model)','') for m in acc_df_live['Model']],
                        rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,110)
    ax.set_title('Accuracy vs NOAA Observed', fontweight='bold')
    ax.legend(fontsize=8)
    for bar in list(b1)+list(b2)+list(b3):
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+1,
                f'{h:.0f}%', ha='center', va='bottom', fontsize=7)
else:
    ax.text(0.5,0.5,'Need ≥2 overlapping\nobserved days for metrics',
            ha='center',va='center',transform=ax.transAxes,
            fontsize=11,color='gray')
    ax.set_title('Accuracy vs NOAA Observed', fontweight='bold')

# Panel 4: error distribution
ax = axes[1,1]
plotted = False
for label, col in pred_cols.items():
    if col not in combined.columns: continue
    sub = combined[combined['kp_noaa_observed'].notna() & combined[col].notna()]
    if len(sub) < 2: continue
    errors = (sub[col] - sub['kp_noaa_observed']).values
    ax.hist(errors, bins=max(5,len(sub)//2), alpha=0.6,
            color=PLOT_COLORS.get(label,'#6366f1'),
            label=label.replace(' (our model)',''), edgecolor='white')
    plotted = True
if plotted:
    ax.axvline(0, color='black', lw=1.2, ls='--', alpha=0.6)
    ax.legend(fontsize=8)
else:
    ax.text(0.5,0.5,'Need ≥2 overlapping\nobserved days',
            ha='center',va='center',transform=ax.transAxes,
            fontsize=11,color='gray')
ax.set_xlabel('Error (Predicted − Observed Kp)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_PLOTS / 'live_noaa_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 65)
print('  SUMMARY')
print('=' * 65)
if not acc_df_live.empty:
    display(acc_df_live.set_index('Model'))
    our = acc_df_live[acc_df_live['Model'].str.contains('our model')]
    noaa_row = acc_df_live[acc_df_live['Model'] == 'NOAA Official Forecast']
    if not our.empty:
        best = our.loc[our['Within ±1.0 Kp %'].idxmax()]
        print(f'\n  Best model      : {best["Model"]}')
        print(f'  Within ±1.0 Kp  : {best["Within ±1.0 Kp %"]:.1f}%')
        print(f'  Aurora event acc: {best["Aurora Acc %"]:.1f}%')
        if not noaa_row.empty:
            delta = best['Within ±1.0 Kp %'] - noaa_row.iloc[0]['Within ±1.0 Kp %']
            print(f'  vs NOAA forecast: {delta:+.1f} pp '
                  f'({"BETTER" if delta>=0 else "WORSE"})')
else:
    print('\n  Not enough overlapping observed days for formal metrics.')
    print('  Predictions are shown in the table and plots above.')
    print('  Re-run tomorrow when more observed data accumulates.')

print(f'\n  Saved → {OUT_EVAL}/live_comparison_full.csv')
print(f'  Saved → {OUT_EVAL}/live_accuracy_summary.csv')
print(f'  Saved → {OUT_PLOTS}/live_noaa_comparison.png')

  FULL LIVE NOAA COMPARISON

[1/5] Fetching live NOAA data...
      mag       : 9524 records
      plasma    : 9780 records
      kp_obs    : 59 records
      kp_fcst   : 81 records
      dst       : 168 records
      f107      : 122 records

[2/5] Parsing into daily averages...
      MAG daily rows   : 8
      Plasma daily rows: 8
      Kp observed days : 8
      Kp forecast days : 3
      Dst daily rows   : 8
      F10.7 daily rows : 41

[3/5] Downloading GFZ for Kp lag features...
  KP: 34,417 rows parsed, 0 skipped  [kp_live_full.txt]
      GFZ ends: 2026-03-24

[4/5] Running trained models on live dates...
      Still missing 1 features (will use col mean):
      ['swpc_kp']
      Target dates     : 11
      Feature rows matched: 7
      XGBoost: 7 predictions
      BiLSTM: 7 predictions

[5/5] Computing metrics and plotting...

Comparison table:
      date  kp_noaa_observed  kp_noaa_forecast  kp_xgboost  kp_bilstm
2026-03-18          1.210000               NaN    0.919635   1.268

,N days,RMSE,MAE,R²,Within ±1.0 Kp %,Within ±0.5 Kp %,Aurora Acc %
Model,,,,,,,
XGBoost (our model),7,0.216,0.182,0.983,100.0,100.0,100.0
BiLSTM (our model),7,1.547,1.179,0.118,57.1,28.6,85.7



  Best model      : XGBoost (our model)
  Within ±1.0 Kp  : 100.0%
  Aurora event acc: 100.0%

  Saved → /content/drive/MyDrive/data/output-data/06_evaluation/live_comparison_full.csv
  Saved → /content/drive/MyDrive/data/output-data/06_evaluation/live_accuracy_summary.csv
  Saved → /content/drive/MyDrive/data/output-data/08_plots/live_noaa_comparison.png


In [ ]:

!pip install cartopy
import cartopy
import urllib.request, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import display

# Install cartopy if needed
try:
    import cartopy
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'cartopy', '-q'])
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

print('=' * 65)
print('  AURORA VISIBILITY & POSITION FORECAST')
print('=' * 65)

def _fetch(url):
    with urllib.request.urlopen(url, timeout=20) as r:
        return _json.loads(r.read().decode())


KP_TO_LAT = {
    0: 66, 1: 64, 2: 62, 3: 60, 4: 58,
    5: 55, 6: 52, 7: 50, 8: 47, 9: 45
}

# Approximate geomagnetic → geographic latitude offset (~3° for UK/Europe)
GEOMAG_TO_GEO_OFFSET = 3.0

def kp_to_visibility_lat(kp):
    """Return equatorward geographic latitude of aurora visibility."""
    kp_int = int(min(9, max(0, round(kp))))
    return KP_TO_LAT[kp_int] - GEOMAG_TO_GEO_OFFSET

def kp_to_storm_label(kp):
    if kp >= 8: return 'G4-G5 (Severe)'
    elif kp >= 7: return 'G3 (Strong)'
    elif kp >= 6: return 'G2 (Moderate)'
    elif kp >= 5: return 'G1 (Minor)'
    elif kp >= 4: return 'Active'
    else: return 'Quiet'

print('\n[1/3] Fetching OVATION aurora grid...')
try:
    ovation_raw = _fetch(
        'https://services.swpc.noaa.gov/json/ovation_aurora_latest.json')

    # Format: {"Forecast Time": "...", "coordinates": [[lon, lat, aurora], ...]}
    obs_time  = ovation_raw.get('Forecast Time', 'unknown')
    coords    = ovation_raw.get('coordinates', [])
    ovation_df = pd.DataFrame(coords, columns=['lon', 'lat', 'aurora'])
    ovation_df = ovation_df[ovation_df['aurora'] > 0]  # drop zero-aurora grid cells
    print(f'      Forecast time    : {obs_time}')
    print(f'      Non-zero cells   : {len(ovation_df):,}')
    print(f'      Max aurora %     : {ovation_df["aurora"].max():.1f}')
    ovation_ok = True
except Exception as e:
    print(f'      OVATION fetch failed: {e}')
    ovation_ok = False

print('\n[2/3] Building visibility forecast table...')
try:
    fcst_raw = _fetch(
        'https://services.swpc.noaa.gov/products/noaa-planetary-k-index-forecast.json')
    fcst_rows = []
    for row in fcst_raw[1:]:
        try:
            fcst_rows.append({
                'time':  pd.to_datetime(row[0]),
                'kp':    float(row[1]),
                'type':  str(row[2]).strip().lower()
            })
        except Exception:
            continue
    fcst_df = pd.DataFrame(fcst_rows)
except Exception as e:
    print(f'      Forecast fetch failed: {e}')
    fcst_df = pd.DataFrame()

# Add our model's Kp predictions to the table
our_kp_rows = []
if 'preds_df' in dir() and len(preds_df) > 0:
    for _, r in preds_df.iterrows():
        our_kp_rows.append({
            'time': pd.to_datetime(r['date']),
            'kp':   float(r['kp_xgboost']),
            'type': 'our_model'
        })
if our_kp_rows:
    our_df = pd.DataFrame(our_kp_rows)
    fcst_df = pd.concat([fcst_df, our_df], ignore_index=True)

# Daily max Kp per source
fcst_df['date'] = fcst_df['time'].dt.floor('D')
noaa_daily = (fcst_df[fcst_df['type'] == 'predicted']
                .groupby('date')['kp'].max().reset_index()
                .rename(columns={'kp': 'kp_noaa_forecast'}))
our_daily  = (fcst_df[fcst_df['type'] == 'our_model']
                .groupby('date')['kp'].max().reset_index()
                .rename(columns={'kp': 'kp_our_model'}))

vis_table = noaa_daily.merge(our_daily, on='date', how='outer').sort_values('date')

# Add visibility latitudes and labels
for src, col in [('NOAA', 'kp_noaa_forecast'), ('Our model', 'kp_our_model')]:
    if col in vis_table.columns:
        vis_table[f'vis_lat_{src.lower().replace(" ","_")}'] = (
            vis_table[col].apply(lambda k: kp_to_visibility_lat(k)
                                 if pd.notna(k) else np.nan))
        vis_table[f'storm_{src.lower().replace(" ","_")}'] = (
            vis_table[col].apply(lambda k: kp_to_storm_label(k)
                                 if pd.notna(k) else ''))

print('\nVisibility forecast table:')
print(vis_table.to_string(index=False))
vis_table.to_csv(OUT_EVAL / 'aurora_visibility_forecast.csv', index=False)


print('\n[3/3] Plotting aurora maps and visibility forecast...')

fig = plt.figure(figsize=(18, 16))
fig.suptitle('Aurora Visibility & Position Forecast', fontsize=15, fontweight='bold')

ax1 = fig.add_subplot(2, 2, 1,
    projection=ccrs.NorthPolarStereo(central_longitude=0))
ax1.set_extent([-180, 180, 30, 90], crs=ccrs.PlateCarree())
ax1.add_feature(cfeature.LAND,       facecolor='#1e293b', zorder=0)
ax1.add_feature(cfeature.OCEAN,      facecolor='#0f172a', zorder=0)
ax1.add_feature(cfeature.COASTLINE,  linewidth=0.4, edgecolor='#475569', zorder=1)
ax1.add_feature(cfeature.BORDERS,    linewidth=0.3, edgecolor='#334155', zorder=1)
ax1.gridlines(color='#334155', linewidth=0.3, alpha=0.5)

if ovation_ok and len(ovation_df) > 0:
    # Filter to northern hemisphere
    nh = ovation_df[ovation_df['lat'] >= 30]
    if len(nh) > 0:
        cmap   = plt.cm.get_cmap('YlOrRd')
        norm   = mcolors.Normalize(vmin=0, vmax=min(100, nh['aurora'].quantile(0.99)))
        sc = ax1.scatter(nh['lon'], nh['lat'], c=nh['aurora'],
                         cmap=cmap, norm=norm, s=2, alpha=0.8,
                         transform=ccrs.PlateCarree(), zorder=2)
        plt.colorbar(sc, ax=ax1, fraction=0.03, pad=0.04,
                     label='Aurora probability (%)')

ax1.set_title(f'NOAA OVATION Aurora (Northern)\n{obs_time}',
              fontsize=10, fontweight='bold')


ax2 = fig.add_subplot(2, 2, 2,
    projection=ccrs.SouthPolarStereo(central_longitude=0))
ax2.set_extent([-180, 180, -90, -30], crs=ccrs.PlateCarree())
ax2.add_feature(cfeature.LAND,       facecolor='#1e293b', zorder=0)
ax2.add_feature(cfeature.OCEAN,      facecolor='#0f172a', zorder=0)
ax2.add_feature(cfeature.COASTLINE,  linewidth=0.4, edgecolor='#475569', zorder=1)
ax2.add_feature(cfeature.BORDERS,    linewidth=0.3, edgecolor='#334155', zorder=1)
ax2.gridlines(color='#334155', linewidth=0.3, alpha=0.5)

if ovation_ok and len(ovation_df) > 0:
    sh = ovation_df[ovation_df['lat'] <= -30]
    if len(sh) > 0:
        cmap = plt.cm.get_cmap('YlOrRd')
        norm = mcolors.Normalize(vmin=0, vmax=min(100, sh['aurora'].quantile(0.99)))
        sc2  = ax2.scatter(sh['lon'], sh['lat'], c=sh['aurora'],
                           cmap=cmap, norm=norm, s=2, alpha=0.8,
                           transform=ccrs.PlateCarree(), zorder=2)
        plt.colorbar(sc2, ax=ax2, fraction=0.03, pad=0.04,
                     label='Aurora probability (%)')

ax2.set_title(f'NOAA OVATION Aurora (Southern)\n{obs_time}',
              fontsize=10, fontweight='bold')


ax3 = fig.add_subplot(2, 2, 3)

if not vis_table.empty:
    dates_v = pd.to_datetime(vis_table['date'])

    if 'kp_noaa_forecast' in vis_table.columns:
        ax3.bar(dates_v - pd.Timedelta(hours=6),
                vis_table['kp_noaa_forecast'], width=0.4,
                color='#64748b', alpha=0.8, label='NOAA Forecast Kp')

    if 'kp_our_model' in vis_table.columns:
        ax3.bar(dates_v + pd.Timedelta(hours=6),
                vis_table['kp_our_model'], width=0.4,
                color='#f59e0b', alpha=0.8, label='Our Model Kp')

    ax3.axhline(5, color='#ef4444', lw=1.2, ls='--', alpha=0.7,
                label='G1 storm (Kp=5)')
    ax3.axhline(7, color='#dc2626', lw=1.2, ls=':',  alpha=0.7,
                label='G3 storm (Kp=7)')

    # Annotate with visibility latitude
    for _, row in vis_table.iterrows():
        kp_val = row.get('kp_noaa_forecast', row.get('kp_our_model', np.nan))
        if pd.notna(kp_val):
            vis_lat = kp_to_visibility_lat(kp_val)
            ax3.text(pd.to_datetime(row['date']),
                     kp_val + 0.15,
                     f'≥{vis_lat:.0f}°N',
                     ha='center', va='bottom', fontsize=8, color='#1e293b')

ax3.set_ylabel('Kp Index')
ax3.set_ylim(0, 10)
ax3.set_title('3-Day Kp Forecast & Visibility Latitude',
              fontsize=11, fontweight='bold')
ax3.legend(fontsize=8)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=20)


ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')

# Key cities and their approximate geographic latitudes
CITIES = {
    'Tromsø, Norway':      69.6,
    'Reykjavik, Iceland':  64.1,
    'Helsinki, Finland':   60.2,
    'Stockholm, Sweden':   59.3,
    'Oslo, Norway':        59.9,
    'Edinburgh, UK':       55.9,
    'Copenhagen, Denmark': 55.7,
    'Amsterdam, NL':       52.4,
    'London, UK':          51.5,
    'Paris, France':       48.9,
    'Berlin, Germany':     52.5,
    'Rome, Italy':         41.9,
    'Madrid, Spain':       40.4,
}

# Get the best Kp forecast across all sources for each day
best_kp_today = np.nan
if not vis_table.empty:
    kp_cols = [c for c in ['kp_noaa_forecast','kp_our_model']
               if c in vis_table.columns]
    if kp_cols:
        best_kp_today = vis_table[kp_cols].max().max()

if pd.notna(best_kp_today):
    min_vis_lat = kp_to_visibility_lat(best_kp_today)
    table_data  = []
    for city, lat in sorted(CITIES.items(), key=lambda x: -x[1]):
        visible = '✓  YES' if lat >= min_vis_lat else '✗  No'
        table_data.append([city, f'{lat:.1f}°N', visible])

    col_labels = ['City', 'Latitude', f'Visible? (Kp={best_kp_today:.1f})']
    tbl = ax4.table(cellText=table_data, colLabels=col_labels,
                     loc='center', cellLoc='left')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.1, 1.4)

    # Colour rows: green = visible, grey = not visible
    for i, (city, lat, _) in enumerate(table_data):
        for j in range(3):
            cell = tbl[i+1, j]
            if lat_val := float(lat.replace('°N','')):
                if lat_val >= min_vis_lat:
                    cell.set_facecolor('#d1fae5')
                    cell.set_text_props(color='#065f46')
                else:
                    cell.set_facecolor('#f1f5f9')
                    cell.set_text_props(color='#64748b')

    # Header style
    for j in range(3):
        tbl[0, j].set_facecolor('#1e293b')
        tbl[0, j].set_text_props(color='white', fontweight='bold')

    ax4.set_title(f'City Visibility (max Kp={best_kp_today:.1f}, '
                  f'≥{min_vis_lat:.0f}°N)',
                  fontsize=11, fontweight='bold', pad=15)
else:
    ax4.text(0.5, 0.5, 'No Kp forecast\navailable',
             ha='center', va='center', transform=ax4.transAxes,
             fontsize=12, color='gray')

plt.tight_layout()
plt.savefig(OUT_PLOTS / 'aurora_visibility_forecast.png',
            dpi=150, bbox_inches='tight')
plt.show()


print('\n' + '=' * 65)
print('  AURORA VISIBILITY SUMMARY')
print('=' * 65)

if pd.notna(best_kp_today):
    print(f'\n  Peak Kp forecast     : {best_kp_today:.1f}  '
          f'({kp_to_storm_label(best_kp_today)})')
    print(f'  Min visibility lat   : ≥{min_vis_lat:.0f}° geographic latitude')
    print(f'\n  Cities where aurora is forecast visible:')
    for city, lat in sorted(CITIES.items(), key=lambda x: -x[1]):
        if lat >= min_vis_lat:
            print(f'    ✓  {city}  ({lat:.1f}°N)')
    print(f'\n  Cities below threshold:')
    for city, lat in sorted(CITIES.items(), key=lambda x: -x[1]):
        if lat < min_vis_lat:
            print(f'    ✗  {city}  ({lat:.1f}°N)')

if ovation_ok:
    print(f'\n  OVATION current max aurora probability: '
          f'{ovation_df["aurora"].max():.1f}%')
    high_aurora = ovation_df[ovation_df['aurora'] >= 50]
    if len(high_aurora) > 0:
        print(f'  High probability (≥50%) cells: {len(high_aurora):,}')
        print(f'  Southernmost high-prob lat: {high_aurora["lat"].min():.1f}°')

print(f'\n  Saved → {OUT_EVAL}/aurora_visibility_forecast.csv')
print(f'  Saved → {OUT_PLOTS}/aurora_visibility_forecast.png')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 24.5 MB/s eta 0:00:00
  AURORA VISIBILITY & POSITION FORECAST

[1/3] Fetching OVATION aurora grid...
      Forecast time    : 2026-03-25T10:28:00Z
      Non-zero cells   : 19,958
      Max aurora %     : 37.0

[2/3] Building visibility forecast table...

Visibility forecast table:
      date  kp_noaa_forecast  kp_our_model  vis_lat_noaa storm_noaa  vis_lat_our_model storm_our_model
2026-03-18               NaN      0.919635           NaN                          61.0           Quiet
2026-03-19               NaN      0.296449           NaN                          63.0           Quiet
2026-03-20               NaN      3.577849           NaN                          55.0           Quiet
2026-03-21               NaN      4.348385           NaN                          55.0          Active
2026-03-22               NaN      5.613758           NaN                          49.0      G1 (Minor)
2026-03-23               NaN      4.121122 

In [ ]:

import urllib.request, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from scipy.ndimage import gaussian_filter

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except ImportError:
    import subprocess
    subprocess.run(['pip','install','cartopy','-q'])
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

print('Fetching OVATION aurora data...')

def _fetch(url):
    with urllib.request.urlopen(url, timeout=20) as r:
        return _json.loads(r.read().decode())

ovation_raw = _fetch(
    'https://services.swpc.noaa.gov/json/ovation_aurora_latest.json')

obs_time = ovation_raw.get('Forecast Time', 'unknown')
coords   = ovation_raw.get('coordinates', [])
ov_df    = pd.DataFrame(coords, columns=['lon','lat','aurora'])

print(f'  Forecast time  : {obs_time}')
print(f'  Grid points    : {len(ov_df):,}')
print(f'  Max aurora %   : {ov_df["aurora"].max():.1f}')
print(f'  Mean aurora %  : {ov_df[ov_df["aurora"]>0]["aurora"].mean():.1f}')


our_kp = np.nan
if 'preds_df' in dir() and 'kp_xgboost' in preds_df.columns:
    our_kp = float(preds_df['kp_xgboost'].iloc[-1])
    print(f'  Our model Kp   : {our_kp:.2f}')

noaa_kp = np.nan
try:
    kp_raw  = _fetch('https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json')
    noaa_kp = float(kp_raw[-1][1])
    print(f'  NOAA current Kp: {noaa_kp:.1f}')
except Exception:
    pass


lon_vals = np.arange(-180, 180, 1)
lat_vals = np.arange(-90,   90, 1)

# Create lookup dict for fast pivot
ov_lookup = {}
for _, row in ov_df.iterrows():
    key = (int(round(row['lon'])), int(round(row['lat'])))
    ov_lookup[key] = row['aurora']

grid = np.zeros((len(lat_vals), len(lon_vals)))
for j, lat in enumerate(lat_vals):
    for i, lon in enumerate(lon_vals):
        grid[j, i] = ov_lookup.get((int(lon), int(lat)), 0.0)

# Light gaussian smooth to remove pixelation — kernel=1 keeps accuracy
grid_smooth = gaussian_filter(grid, sigma=0.8)

KP_TO_LAT = {0:66,1:64,2:62,3:60,4:58,5:55,6:52,7:50,8:47,9:45}
def kp_oval_lat(kp):
    k = int(min(9, max(0, round(kp))))
    return KP_TO_LAT[k] - 3.0  # geo offset

# Plot 
fig = plt.figure(figsize=(18, 8), facecolor='#020817')

# Northern hemisphere
ax1 = fig.add_subplot(1, 2, 1,
    projection=ccrs.NorthPolarStereo(central_longitude=0))
ax1.set_extent([-180, 180, 30, 90], crs=ccrs.PlateCarree())
ax1.set_facecolor('#020817')

# Base map
ax1.add_feature(cfeature.LAND,
    facecolor='#0f172a', edgecolor='none', zorder=1)
ax1.add_feature(cfeature.OCEAN,
    facecolor='#020817', edgecolor='none', zorder=0)
ax1.add_feature(cfeature.COASTLINE,
    linewidth=0.5, edgecolor='#334155', zorder=3)
ax1.add_feature(cfeature.BORDERS,
    linewidth=0.3, edgecolor='#1e3a4a', zorder=3)
ax1.gridlines(color='#1e293b', linewidth=0.4, alpha=0.6,
              xlocs=range(-180,181,30), ylocs=range(30,91,10))

# OVATION aurora probability — northern hemisphere only
nh_mask = (lat_vals >= 30)
nh_grid = grid_smooth[nh_mask, :]
nh_lats = lat_vals[nh_mask]

# Custom colourmap: transparent→green→white (aurora colours)
from matplotlib.colors import LinearSegmentedColormap
aurora_cmap = LinearSegmentedColormap.from_list(
    'aurora',
    [(0.0, (0.0,  0.0,  0.0,  0.0)),   # transparent
     (0.2, (0.0,  0.5,  0.2,  0.4)),   # faint green
     (0.5, (0.1,  0.9,  0.3,  0.7)),   # green
     (0.8, (0.5,  1.0,  0.7,  0.9)),   # bright green
     (1.0, (0.9,  1.0,  1.0,  1.0))],  # white
    N=256)

lon_mesh, lat_mesh = np.meshgrid(lon_vals, nh_lats)
im1 = ax1.pcolormesh(lon_mesh, lat_mesh, nh_grid,
                      cmap=aurora_cmap,
                      norm=mcolors.Normalize(vmin=0, vmax=100),
                      transform=ccrs.PlateCarree(),
                      zorder=2, alpha=0.9)

# NOAA oval boundary circle
if not np.isnan(noaa_kp):
    oval_lat = kp_oval_lat(noaa_kp)
    theta    = np.linspace(0, 2*np.pi, 360)
    oval_lons = np.degrees(theta)
    oval_lats = np.full_like(oval_lons, oval_lat)
    ax1.plot(oval_lons, oval_lats,
             color='#64748b', lw=1.2, ls='--', alpha=0.7,
             transform=ccrs.PlateCarree(), zorder=4,
             label=f'NOAA oval (Kp={noaa_kp:.0f})')

# Our model oval boundary
if not np.isnan(our_kp):
    oval_lat_our = kp_oval_lat(our_kp)
    ax1.plot(oval_lons, np.full_like(oval_lons, oval_lat_our),
             color='#f59e0b', lw=1.5, ls='-', alpha=0.85,
             transform=ccrs.PlateCarree(), zorder=4,
             label=f'Our model oval (Kp={our_kp:.1f})')

cb1 = plt.colorbar(im1, ax=ax1, fraction=0.03, pad=0.05,
                   orientation='horizontal')
cb1.set_label('Aurora probability (%)', color='#94a3b8', fontsize=9)
cb1.ax.xaxis.set_tick_params(color='#94a3b8')
plt.setp(cb1.ax.xaxis.get_ticklabels(), color='#94a3b8', fontsize=8)
ax1.set_title(f'Northern Hemisphere Aurora\n{obs_time}',
              color='white', fontsize=11, fontweight='bold', pad=8)
ax1.legend(loc='lower left', fontsize=8,
           facecolor='#0f172a', edgecolor='#334155',
           labelcolor='white')

# Southern hemisphere
ax2 = fig.add_subplot(1, 2, 2,
    projection=ccrs.SouthPolarStereo(central_longitude=0))
ax2.set_extent([-180, 180, -90, -30], crs=ccrs.PlateCarree())
ax2.set_facecolor('#020817')

ax2.add_feature(cfeature.LAND,
    facecolor='#0f172a', edgecolor='none', zorder=1)
ax2.add_feature(cfeature.OCEAN,
    facecolor='#020817', edgecolor='none', zorder=0)
ax2.add_feature(cfeature.COASTLINE,
    linewidth=0.5, edgecolor='#334155', zorder=3)
ax2.add_feature(cfeature.BORDERS,
    linewidth=0.3, edgecolor='#1e3a4a', zorder=3)
ax2.gridlines(color='#1e293b', linewidth=0.4, alpha=0.6,
              xlocs=range(-180,181,30), ylocs=range(-90,-29,10))

sh_mask = (lat_vals <= -30)
sh_grid = grid_smooth[sh_mask, :]
sh_lats = lat_vals[sh_mask]
lon_mesh_s, lat_mesh_s = np.meshgrid(lon_vals, sh_lats)

im2 = ax2.pcolormesh(lon_mesh_s, lat_mesh_s, sh_grid,
                      cmap=aurora_cmap,
                      norm=mcolors.Normalize(vmin=0, vmax=100),
                      transform=ccrs.PlateCarree(),
                      zorder=2, alpha=0.9)

if not np.isnan(noaa_kp):
    ax2.plot(oval_lons, np.full_like(oval_lons, -oval_lat),
             color='#64748b', lw=1.2, ls='--', alpha=0.7,
             transform=ccrs.PlateCarree(), zorder=4)
if not np.isnan(our_kp):
    ax2.plot(oval_lons, np.full_like(oval_lons, -oval_lat_our),
             color='#f59e0b', lw=1.5, ls='-', alpha=0.85,
             transform=ccrs.PlateCarree(), zorder=4)

cb2 = plt.colorbar(im2, ax=ax2, fraction=0.03, pad=0.05,
                   orientation='horizontal')
cb2.set_label('Aurora probability (%)', color='#94a3b8', fontsize=9)
cb2.ax.xaxis.set_tick_params(color='#94a3b8')
plt.setp(cb2.ax.xaxis.get_ticklabels(), color='#94a3b8', fontsize=8)
ax2.set_title(f'Southern Hemisphere Aurora\n{obs_time}',
              color='white', fontsize=11, fontweight='bold', pad=8)

caption = (
    f'Colour = NOAA OVATION probability (0–100%).  '
    f'Dashed circle = NOAA oval boundary (Kp={noaa_kp:.0f}).  '
    f'Solid circle = our model oval boundary (Kp={our_kp:.1f}).  '
    f'Accuracy: OVATION ±2° latitude, ±30 min timing.'
)
fig.text(0.5, 0.01, caption, ha='center', color='#64748b',
         fontsize=8, wrap=True)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(OUT_PLOTS / 'aurora_probability_map.png',
            dpi=200, bbox_inches='tight',
            facecolor='#020817')
plt.show()

print(f'\n  Saved → {OUT_PLOTS}/aurora_probability_map.png')
print(f'  Data source: NOAA OVATION Prime model')
print(f'  Oval source: Feldstein–Starkov empirical boundary')
print(f'  Note: image shows aurora *probability*, not appearance')

Fetching OVATION aurora data...
  Forecast time  : 2026-03-25T10:28:00Z
  Grid points    : 65,160
  Max aurora %   : 37.0
  Mean aurora %  : 8.8
  Our model Kp   : 3.13
  NOAA current Kp: 5.3

  Saved → /content/drive/MyDrive/data/output-data/08_plots/aurora_probability_map.png
  Data source: NOAA OVATION Prime model
  Oval source: Feldstein–Starkov empirical boundary
  Note: image shows aurora *probability*, not appearance


In [ ]:


import urllib.request, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import math

def _fetch(url):
    with urllib.request.urlopen(url, timeout=20) as r:
        return _json.loads(r.read().decode())

print('Fetching forecast data...')

Fetch Kp
try:
    kp_obs_raw  = _fetch('https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json')
    kp_fcst_raw = _fetch('https://services.swpc.noaa.gov/products/noaa-planetary-k-index-forecast.json')
except Exception as e:
    print(f'ERROR fetching Kp data: {e}'); raise

# Parse observed (last ~3 days)
obs_rows = []
for row in kp_obs_raw[1:]:
    try:
        obs_rows.append({'time': pd.to_datetime(row[0]),
                         'kp': float(row[1]), 'type': 'observed'})
    except Exception: continue

# Parse forecast
fcst_rows = []
for row in kp_fcst_raw[1:]:
    try:
        fcst_rows.append({'time': pd.to_datetime(row[0]),
                          'kp': float(row[1]),
                          'type': str(row[2]).strip().lower()})
    except Exception: continue

all_kp = pd.DataFrame(obs_rows + fcst_rows)
all_kp['date'] = all_kp['time'].dt.floor('D')

# Daily max per type
obs_daily  = (all_kp[all_kp['type']=='observed']
                .groupby('date')['kp'].max().reset_index()
                .rename(columns={'kp':'kp_observed'}))
fcst_daily = (all_kp[all_kp['type']=='predicted']
                .groupby('date')['kp'].max().reset_index()
                .rename(columns={'kp':'kp_noaa_forecast'}))

# Our model predictions from previous cell
our_rows = []
if 'preds_df' in dir() and len(preds_df) > 0:
    for _, r in preds_df.iterrows():
        our_rows.append({'date': pd.to_datetime(r['date']).floor('D'),
                         'kp_our_model': float(r['kp_xgboost'])})
our_daily = pd.DataFrame(our_rows) if our_rows else pd.DataFrame(
    columns=['date','kp_our_model'])
if not our_daily.empty:
    our_daily = our_daily.groupby('date')['kp_our_model'].max().reset_index()

# Merge all
calendar = (obs_daily
    .merge(fcst_daily,  on='date', how='outer')
    .merge(our_daily,   on='date', how='outer')
    .sort_values('date').reset_index(drop=True))
calendar['date'] = pd.to_datetime(calendar['date'])

# Best available Kp per day (observed > our model > noaa forecast)
def best_kp(row):
    if pd.notna(row.get('kp_observed')):    return row['kp_observed'],    'observed'
    if pd.notna(row.get('kp_our_model')):   return row['kp_our_model'],   'our model'
    if pd.notna(row.get('kp_noaa_forecast')):return row['kp_noaa_forecast'],'noaa fcst'
    return np.nan, 'n/a'

calendar[['kp_best','kp_source']] = pd.DataFrame(
    [best_kp(r) for _, r in calendar.iterrows()],
    columns=['kp_best','kp_source'])

calendar = calendar.dropna(subset=['kp_best']).reset_index(drop=True)

# Keep only last 2 observed + up to 5 forecast = 7 days max
calendar = calendar.tail(7).reset_index(drop=True)
print(f'  Calendar days: {len(calendar)}')
print(calendar[['date','kp_best','kp_source']].to_string(index=False))

def moon_phase(date):
    # Approximate lunation — known new moon reference: 2000-01-06
    ref  = pd.Timestamp('2000-01-06')
    days = (pd.Timestamp(date) - ref).days % 29.53
    phase = days / 29.53
    # Viewing score: new moon (0%) = best, full moon (100%) = worst
    moon_illumination = abs(math.cos(phase * 2 * math.pi)) * 100  # 0=new,100=full
    viewing_pct = round(100 - moon_illumination)
    if phase < 0.125 or phase > 0.875:   label = '🌑 New'
    elif phase < 0.25:                    label = '🌒 Crescent'
    elif phase < 0.375:                   label = '🌓 Quarter'
    elif phase < 0.5:                     label = '🌔 Gibbous'
    elif phase < 0.625:                   label = '🌕 Full'
    elif phase < 0.75:                    label = '🌖 Gibbous'
    elif phase < 0.875:                   label = '🌗 Quarter'
    else:                                  label = '🌘 Crescent'
    return label, viewing_pct

def kp_props(kp):
    if kp >= 8:   return '#ef4444', 'SEVERE',    5
    elif kp >= 7: return '#f97316', 'STRONG',    5
    elif kp >= 6: return '#f59e0b', 'MODERATE',  5
    elif kp >= 5: return '#eab308', 'STORM',     5
    elif kp >= 4: return '#84cc16', 'ACTIVE',    4
    elif kp >= 3: return '#22c55e', 'UNSETTLED', 3
    elif kp >= 2: return '#10b981', 'QUIET',     2
    else:          return '#6b7280', 'QUIET',     1

# Draw calendar
n_days   = len(calendar)
card_w   = 1.6
card_h   = 3.2
gap      = 0.18
fig_w    = n_days * (card_w + gap) + 0.4
fig_h    = card_h + 0.8

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')
ax.set_xlim(0, fig_w)
ax.set_ylim(0, fig_h)
ax.axis('off')

for i, row in calendar.iterrows():
    x0  = 0.2 + i * (card_w + gap)
    y0  = 0.3
    kp  = row['kp_best']
    src = row['kp_source']
    color, storm_label, _ = kp_props(kp)
    moon_label, moon_pct  = moon_phase(row['date'])
    date_obj = pd.Timestamp(row['date'])
    is_obs   = src == 'observed'

    # Card background
    card_bg = '#1e293b' if not is_obs else '#172032'
    card = FancyBboxPatch((x0, y0), card_w, card_h,
                           boxstyle='round,pad=0.04',
                           facecolor=card_bg,
                           edgecolor='#334155',
                           linewidth=1.0, zorder=1)
    ax.add_patch(card)

    # "PAST" watermark for observed days
    if is_obs:
        ax.text(x0 + card_w/2, y0 + card_h - 0.18,
                'OBSERVED', ha='center', va='top',
                fontsize=5.5, color='#475569',
                fontfamily='monospace', zorder=3)

    # Month label
    ax.text(x0 + card_w/2, y0 + card_h - 0.05,
            date_obj.strftime('%b').upper(),
            ha='center', va='top', fontsize=7,
            color='#94a3b8', fontweight='bold', zorder=3)

    # Day number — large
    ax.text(x0 + card_w/2, y0 + card_h - 0.52,
            date_obj.strftime('%-d'),
            ha='center', va='top', fontsize=28,
            color='white', fontweight='bold', zorder=3)

    # Weekday
    ax.text(x0 + card_w/2, y0 + card_h - 0.88,
            date_obj.strftime('%a'),
            ha='center', va='top', fontsize=8,
            color='#64748b', zorder=3)

    # Kp value — coloured
    kp_label = f'Kp {kp:.0f}'
    ax.text(x0 + card_w/2, y0 + card_h - 1.25,
            kp_label,
            ha='center', va='top', fontsize=13,
            color=color, fontweight='bold', zorder=3)

    # Coloured dot under Kp
    dot = plt.Circle((x0 + card_w/2, y0 + card_h - 1.52),
                      0.06, color=color, zorder=3)
    ax.add_patch(dot)

    # Storm level label
    ax.text(x0 + card_w/2, y0 + card_h - 1.72,
            storm_label,
            ha='center', va='top', fontsize=7.5,
            color=color, fontweight='bold',
            alpha=0.9, zorder=3)

    # Divider line
    ax.plot([x0 + 0.12, x0 + card_w - 0.12],
            [y0 + card_h - 1.88, y0 + card_h - 1.88],
            color='#334155', lw=0.6, zorder=3)

    # Moon phase label
    moon_emoji = moon_label.split()[0]
    moon_name  = moon_label.split()[1]
    ax.text(x0 + card_w/2, y0 + card_h - 2.08,
            moon_emoji,
            ha='center', va='top', fontsize=14, zorder=3)
    ax.text(x0 + card_w/2, y0 + card_h - 2.34,
            moon_name,
            ha='center', va='top', fontsize=6.5,
            color='#94a3b8', zorder=3)

    # Viewing % (sky darkness proxy — higher = better dark sky)
    view_color = ('#22c55e' if moon_pct >= 70 else
                  '#eab308' if moon_pct >= 40 else '#ef4444')
    ax.text(x0 + card_w/2, y0 + card_h - 2.62,
            f'{moon_pct}% dark sky',
            ha='center', va='top', fontsize=6.5,
            color=view_color, zorder=3)

    # Data source tag
    src_color = '#f59e0b' if 'model' in src else '#64748b'
    ax.text(x0 + card_w/2, y0 + 0.12,
            src.upper(),
            ha='center', va='bottom', fontsize=5.5,
            color=src_color, alpha=0.8,
            fontfamily='monospace', zorder=3)


ax.text(fig_w/2, fig_h - 0.08,
        'Aurora Forecast Calendar',
        ha='center', va='top', fontsize=13,
        color='white', fontweight='bold')

# Legend
legend_items = [
    mpatches.Patch(color='#f59e0b', label='Our model prediction'),
    mpatches.Patch(color='#64748b', label='NOAA forecast'),
    mpatches.Patch(color='#1e293b', label='Observed / past'),
]
ax.legend(handles=legend_items, loc='lower right',
          facecolor='#1e293b', edgecolor='#334155',
          labelcolor='#94a3b8', fontsize=7,
          bbox_to_anchor=(0.99, 0.01))

plt.tight_layout(pad=0.3)
plt.savefig(OUT_PLOTS / 'aurora_calendar_forecast.png',
            dpi=180, bbox_inches='tight',
            facecolor='#0f172a')
plt.show()

print(f'\n  Saved → {OUT_PLOTS}/aurora_calendar_forecast.png')


print('\n' + '=' * 55)
print('  AURORA FORECAST CALENDAR')
print('=' * 55)
for _, row in calendar.iterrows():
    kp = row['kp_best']
    color_str, label, _ = kp_props(kp)
    moon_l, moon_p = moon_phase(row['date'])
    our = f"  |  Our model: Kp {row['kp_our_model']:.1f}" \
          if pd.notna(row.get('kp_our_model')) else ''
    noaa = f"  |  NOAA: Kp {row['kp_noaa_forecast']:.1f}" \
           if pd.notna(row.get('kp_noaa_forecast')) else ''
    print(f"  {pd.Timestamp(row['date']).strftime('%a %b %-d'):10}"
          f"  Kp {kp:.1f}  {label:<10}"
          f"  {moon_l}  Dark sky: {moon_p}%"
          f"{our}{noaa}")

Fetching forecast data...
  Calendar days: 7
      date  kp_best kp_source
2026-03-22     6.67  observed
2026-03-23     5.00  observed
2026-03-24     3.67  observed
2026-03-25     5.33  observed
2026-03-26     3.67 noaa fcst
2026-03-27     2.67 noaa fcst
2026-03-28     1.67 noaa fcst

  Saved → /content/drive/MyDrive/data/output-data/08_plots/aurora_calendar_forecast.png

  AURORA FORECAST CALENDAR
  Sun Mar 22  Kp 6.7  MODERATE    🌒 Crescent  Dark sky: 39%  |  Our model: Kp 5.6
  Mon Mar 23  Kp 5.0  STORM       🌒 Crescent  Dark sky: 57%  |  Our model: Kp 4.1
  Tue Mar 24  Kp 3.7  UNSETTLED   🌒 Crescent  Dark sky: 77%  |  Our model: Kp 3.1
  Wed Mar 25  Kp 5.3  STORM       🌒 Crescent  Dark sky: 98%
  Thu Mar 26  Kp 3.7  UNSETTLED   🌓 Quarter  Dark sky: 81%  |  NOAA: Kp 3.7
  Fri Mar 27  Kp 2.7  QUIET       🌓 Quarter  Dark sky: 61%  |  NOAA: Kp 2.7
  Sat Mar 28  Kp 1.7  QUIET       🌓 Quarter  Dark sky: 42%  |  NOAA: Kp 1.7


In [ ]:

import os
import shutil
from pathlib import Path
from google.colab import files as colab_files


LOCAL_DEST = r"C:\Users\shahr\Downloads\data-outputs"
.
FOLDER_MAP = {
    OUT_CLEAN:  "01_cleaned_data",
    OUT_FEAT:   "02_feature_data",
    OUT_GA:     "03_ga_feature_selection",
    OUT_XGB:    "04_xgboost",
    OUT_LSTM:   "05_lstm",
    OUT_EVAL:   "06_evaluation",
    OUT_MODELS: "07_models",
    OUT_PLOTS:  "08_plots",
}


INCLUDE_EXTENSIONS = {
    ".csv", ".json", ".txt", ".png", ".gif", ".pkl", ".keras", ".joblib"
}


all_files = []   

for drive_folder, local_sub in FOLDER_MAP.items():
    drive_folder = Path(drive_folder)
    if not drive_folder.exists():
        print(f"  ⚠ Skipping (not found): {drive_folder}")
        continue
    for f in sorted(drive_folder.rglob("*")):
        if not f.is_file():
            continue
        if INCLUDE_EXTENSIONS and f.suffix.lower() not in INCLUDE_EXTENSIONS:
            continue
        all_files.append((f, local_sub))

print(f"\n  Found {len(all_files)} files to download across {len(FOLDER_MAP)} folders.")
print(f"  Destination: {LOCAL_DEST}\n")


import zipfile
import time

STAGING_DIR = Path("/content/download_staging")
STAGING_DIR.mkdir(exist_ok=True)

ZIP_NAME = f"aurora_output_data_{time.strftime('%Y%m%d_%H%M%S')}.zip"
ZIP_PATH = STAGING_DIR / ZIP_NAME

copied   = 0
skipped  = 0
errors   = []

print("  Packaging files…")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for drive_path, local_sub in all_files:
        try:
            # Archive path mirrors the local destination structure
            arc_path = os.path.join(local_sub, drive_path.name)
            zf.write(str(drive_path), arcname=arc_path)
            copied += 1
            if copied % 10 == 0:
                print(f"    {copied}/{len(all_files)} files packaged…")
        except Exception as e:
            skipped += 1
            errors.append(f"  ✗ {drive_path.name}: {e}")

zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f"\n  ✓ Archive ready: {ZIP_NAME}")
print(f"    Files packaged : {copied}")
print(f"    Skipped/errors : {skipped}")
print(f"    Archive size   : {zip_size_mb:.1f} MB")
if errors:
    print("\n  Errors:")
    for e in errors:
        print(e
print(f"""
  ┌─────────────────────────────────────────────────────────────┐
  │  INSTALLATION INSTRUCTIONS                                  │
  │                                                             │
  │  1. The download will start automatically below.            │
  │  2. Save the zip to:                                        │
  │       {LOCAL_DEST:<45}│
  │  3. Extract it there — it will create sub-folders:         │
  │       01_cleaned_data\\                                      │
  │       02_feature_data\\                                      │
  │       03_ga_feature_selection\\                              │
  │       04_xgboost\\                                           │
  │       05_lstm\\                                              │
  │       06_evaluation\\                                        │
  │       07_models\\                                            │
  │       08_plots\\                                             │
  │  4. The server reads from this exact path automatically.    │
  └─────────────────────────────────────────────────────────────┘
""")


KEY_FILES = [
    (OUT_EVAL  / "ensemble_predictions.csv",    "ensemble predictions"),
    (OUT_EVAL  / "model_comparison.csv",         "model comparison metrics"),
    (OUT_EVAL  / "aurora_visibility_forecast.csv","visibility forecast"),
    (OUT_PLOTS / "aurora_calendar_forecast.png", "calendar forecast image"),
    (OUT_PLOTS / "aurora_probability_map.png",   "probability map image"),
    (OUT_PLOTS / "live_noaa_comparison.png",     "live NOAA comparison"),
    (OUT_XGB   / "xgboost_test_predictions.csv", "XGBoost predictions"),
    (OUT_EVAL  / "live_comparison_full.csv",     "full live comparison"),
    (OUT_EVAL  / "live_accuracy_summary.csv",    "accuracy summary"),
]

print("  Key files available as individual downloads:")
individual_found = []
for fpath, label in KEY_FILES:
    fpath = Path(fpath)
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f"    ✓  {fpath.name:<45} ({size_kb:.0f} KB)  — {label}")
        individual_found.append(fpath)
    else:
        print(f"    –  {fpath.name:<45}  (not generated yet)")


print(f"\n  ⬇  Downloading zip archive ({zip_size_mb:.1f} MB)…")
colab_files.download(str(ZIP_PATH))

# Small delay, then offer individual key files
import time
time.sleep(1)

if individual_found:
    print(f"\n  ⬇  Downloading {len(individual_found)} key files individually…")
    for fpath in individual_found:
        try:
            colab_files.download(str(fpath))
            print(f"    ↓  {fpath.name}")
            time.sleep(0.3)
        except Exception as e:
            print(f"    ✗  {fpath.name}: {e}")

print(f"""
  ✓ All downloads triggered.

  Once extracted to {LOCAL_DEST}
  the server will automatically read from that location.

  To check what's available, open:
    http://localhost:5000/pipeline/files
""")


  Found 41 files to download across 8 folders.
  Destination: C:\Users\shahr\Downloads\data-outputs

  Packaging files…
    10/41 files packaged…
    20/41 files packaged…
    30/41 files packaged…
    40/41 files packaged…

  ✓ Archive ready: aurora_output_data_20260325_104722.zip
    Files packaged : 41
    Skipped/errors : 0
    Archive size   : 17.8 MB

  ┌─────────────────────────────────────────────────────────────┐
  │  INSTALLATION INSTRUCTIONS                                  │
  │                                                             │
  │  1. The download will start automatically below.            │
  │  2. Save the zip to:                                        │
  │       C:\Users\shahr\Downloads\data-outputs        │
  │  3. Extract it there — it will create sub-folders:         │
  │       01_cleaned_data\                                      │
  │       02_feature_data\                                      │
  │       03_ga_feature_selection\                     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  ⬇  Downloading 9 key files individually…


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

    ↓  live_comparison_full.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    ↓  live_accuracy_summary.csv

  ✓ All downloads triggered.

  Once extracted to C:\Users\shahr\Downloads\data-outputs
  the server will automatically read from that location.

  To check what's available, open:
    http://localhost:5000/pipeline/files

